In [ ]:
%%configure
{
  "defaultLakehouse": {
    "name": "academic_early_warning"
  }
}


In [ ]:
# Academic Early Warning System: load all Lakehouse input tables
# Do not use Load to Tables for the replication Quickstart.
# Extract the simulated-data ZIP on your computer. In Lakehouse Files choose Upload folder.
# Upload the whole extracted folder, retaining all files and nesting, with only one root folder.
# Expected: Files/academic-early-warning-simulated-data/{source_parquet/, features/, manifest.json, summary.json}
# Cell 1 configures the default Lakehouse by name in this notebook's workspace.
# Stop an existing notebook session before Run all so cell 1 takes effect.
# Runtime 1.3: Python 3.11, Spark 3.5, Delta 3.2. Use the existing spark session; no installs.
# MLflow is not required for this table-only loader. Delta uses Fabric-provided Python/Spark classes.
# Run all once: verify the folder and create all 27 correctly named Delta input tables.
# Identical existing tables are verified and skipped. Conflicts stop before new table creation.
# No files or tables are deleted or overwritten. No training, gold outputs or publication.
# Optional: run 01 to train and publish, then 02 to explore that completed publication.
print('Upload the extracted folder once to Files, set SOURCE_FOLDER, then Run all. No per-table wizard.')
import sys
import types
EMBEDDED_MODULES = {'aews': '"""Academic Early Warning System: synthetic replication starter."""\n', 'aews.core': '"""Selected source contracts, synthetic history and point-in-time features."""\n', 'aews.core.contract': 'import ipaddress\nimport json\nimport uuid\n\nimport numpy as np\nimport pandas as pd\n\nHISTORY = ("effective_at", "recorded_at", "revision")\n\n\ndef frame(table, records, contract, history=False):\n    specs = contract["tables"][table]["columns"]\n    columns = list(specs) + (list(HISTORY) if history else [])\n    result = pd.DataFrame.from_records(records, columns=columns)\n    for name, spec in specs.items():\n        kind = spec["type"]\n        if kind in ("int64", "int32"):\n            present = result[name].dropna()\n            if any(isinstance(v, (float, np.floating)) and v != int(v) for v in present):\n                raise ValueError(f"Fractional identifier: {table}.{name}")\n            result[name] = result[name].astype("Int64" if kind == "int64" else "Int32")\n        elif kind == "float64":\n            result[name] = result[name].astype("float64")\n        elif kind == "bool":\n            result[name] = result[name].astype("boolean")\n        elif kind == "datetime":\n            result[name] = pd.to_datetime(result[name], utc=True).dt.floor("us")\n        elif kind == "string":\n            result[name] = result[name].astype("string")\n    if history:\n        for name in HISTORY[:2]:\n            result[name] = pd.to_datetime(result[name], utc=True).dt.floor("us")\n        result["revision"] = result["revision"].astype("int32")\n    return result\n\n\ndef as_of(revisions, cutoff):\n    available = revisions.loc[\n        (revisions.effective_at <= cutoff) & (revisions.recorded_at <= cutoff)\n    ]\n    # Event order and availability are independent: a late-arriving old revision\n    # must not displace an already-observed newer source revision.\n    return available.sort_values(["id", "effective_at", "revision"]).drop_duplicates("id", keep="last").reset_index(drop=True)\n\n\ndef validate_table(table, data, contract, references=None, history=False):\n    specs = contract["tables"][table]["columns"]\n    required_columns = list(specs) + (list(HISTORY) if history else [])\n    if list(data.columns) != required_columns:\n        raise ValueError(f"Column projection mismatch: {table}")\n    key = ["id", "revision"] if history else ["id"]\n    if data.duplicated(key).any():\n        raise ValueError(f"Duplicate key in {table}")\n    for name, spec in specs.items():\n        values = data[name]\n        present = values.dropna()\n        if not spec["nullable"] and values.isna().any():\n            raise ValueError(f"Null mandatory field: {table}.{name}")\n        kind = spec["type"]\n        if kind.startswith("int") and str(values.dtype).lower() != kind:\n            raise ValueError(f"Wrong integer width: {table}.{name}: {values.dtype}")\n        if kind == "float64" and (str(values.dtype) != "float64" or not np.isfinite(present).all()):\n            raise ValueError(f"Nonfinite/wrong numeric type: {table}.{name}")\n        if kind == "bool" and str(values.dtype) != "boolean":\n            raise ValueError(f"Wrong boolean type: {table}.{name}")\n        if kind == "string" and str(values.dtype) != "string":\n            raise ValueError(f"Wrong string type: {table}.{name}")\n        if kind == "datetime" and not isinstance(values.dtype, pd.DatetimeTZDtype):\n            raise ValueError(f"Missing UTC timestamp: {table}.{name}")\n        if kind == "array<string>":\n            if any(not isinstance(v, list) or any(not isinstance(x, str) for x in v) for v in present):\n                raise ValueError(f"Invalid array: {table}.{name}")\n            if spec["enum"] and any(not set(v) <= set(spec["enum"]) for v in present):\n                raise ValueError(f"Invalid array enum: {table}.{name}")\n        elif spec["enum"] and not present.isin(spec["enum"]).all():\n            raise ValueError(f"Invalid enum: {table}.{name}")\n        if spec["max_length"] and present.map(len).gt(spec["max_length"]).any():\n            raise ValueError(f"String exceeds public maximum: {table}.{name}")\n        if spec["format"] == "uuid":\n            for value in present:\n                if str(uuid.UUID(value)) != value:\n                    raise ValueError(f"Noncanonical UUID in {table}")\n        if spec["format"] == "ip":\n            for value in present:\n                ipaddress.ip_address(value)\n        ref = spec["reference"]\n        if references is not None and ref:\n            # CD2 explicitly permits external web-log users; they are counted\n            # and excluded from learner features, not manufactured as local users.\n            if table == "web_logs" and name == "user_id":\n                continue\n            if ref not in references:\n                raise ValueError(f"Missing reference target {ref}")\n            if not present.isin(references[ref]["id"]).all():\n                raise ValueError(f"Orphan reference: {table}.{name} -> {ref}")\n    if history:\n        if data[list(HISTORY)].isna().any().any():\n            raise ValueError(f"Null revision metadata: {table}")\n        if (data.recorded_at < data.effective_at).any():\n            raise ValueError(f"Observation precedes event: {table}")\n    return len(data)\n\n\ndef json_safe(value):\n    if isinstance(value, dict):\n        return {str(k): json_safe(v) for k, v in value.items()}\n    if isinstance(value, (list, tuple, np.ndarray)):\n        return [json_safe(v) for v in value]\n    if isinstance(value, (pd.Timestamp,)):\n        return value.isoformat()\n    if isinstance(value, (np.integer,)):\n        return int(value)\n    if isinstance(value, (np.floating, float)):\n        return float(value) if np.isfinite(value) else None\n    if isinstance(value, (np.bool_,)):\n        return bool(value)\n    if value is pd.NA or value is pd.NaT:\n        return None\n    return value\n\n\ndef dumps(value):\n    return json.dumps(json_safe(value), sort_keys=True, allow_nan=False)\n', 'aews.core.simulation': 'from dataclasses import dataclass\nimport hashlib\nimport uuid\n\nimport numpy as np\nimport pandas as pd\n\nfrom aews.core.contract import HISTORY, as_of, frame, validate_table\n\nDAY = pd.Timedelta(days=1)\nORIGIN = pd.Timestamp("2023-01-09", tz="UTC")\nSTATIC = ("accounts", "users", "enrollment_terms")\n\n\n@dataclass\nclass CourseBatch:\n    term: int\n    course_id: int\n    start: pd.Timestamp\n    end: pd.Timestamp\n    publication: pd.Timestamp\n    revisions: dict\n    withdrawal_events: pd.DataFrame\n    qualification: pd.DataFrame\n\n    def source(self, table, cutoff=None):\n        return as_of(self.revisions[table], cutoff if cutoff is not None else self.publication)\n\n\ndef validate_config(config):\n    size = config["profiles"][config["mode"]]\n    if config["terms"] != 5 or config["term_days"] != 84:\n        raise ValueError("This version requires five 84-day terms.")\n    if not set(config["cutoff_weeks"]) <= {2, 4, 6} or 4 not in config["cutoff_weeks"]:\n        raise ValueError("Week four is required; optional cutoffs are two and six.")\n    if config["current_week"] not in config["cutoff_weeks"]:\n        raise ValueError("Current week must be an enabled cutoff.")\n    if not 1 <= size["courses_per_term"] <= 100 or not 1 <= size["assignments_per_course"] <= 30:\n        raise ValueError("Course/assignment count exceeds bounded ID space.")\n    if not 1 <= size["enrollments_per_course"] <= min(500, int(size["students"] * .8)):\n        raise ValueError("Invalid enrollment sample or bounded course chunk.")\n    if not 1 <= size["sections_per_course"] <= 9:\n        raise ValueError("Section count exceeds bounded ID space.")\n    if size["students"] > 5000:\n        raise ValueError("This bounded build is bounded to 5,000 students.")\n    return size\n\n\ndef dimensions(config, contract):\n    size = validate_config(config)\n    created = ORIGIN - 120 * DAY\n    def dated(row, state):\n        return dict(row, created_at=created, updated_at=created, workflow_state=state)\n    users = [dated({"id": i, "name": f"Fictional Learner {i:05d}"}, "registered")\n             for i in range(1, size["students"] + 1)]\n    users += [dated({"id": size["students"] + i, "name": f"Fictional Instructor {i:03d}"}, "registered")\n              for i in range(1, size["courses_per_term"] + 1)]\n    terms = [dated({"id": t, "name": f"Synthetic Term {t}", "start_at": ORIGIN + (t - 1) * 120 * DAY,\n                    "end_at": ORIGIN + ((t - 1) * 120 + 84) * DAY}, "active") for t in range(1, 6)]\n    rows = {"accounts": [dated({"id": 1, "name": "Synthetic Learning Sandbox"}, "active")],\n            "users": users, "enrollment_terms": terms}\n    return {name: frame(name, records, contract) for name, records in rows.items()}\n\n\ndef _sigmoid(x):\n    return 1 / (1 + np.exp(-np.clip(x, -30, 30)))\n\n\ndef generate_course(config, contract, term, course_number):\n    size = validate_config(config)\n    seed = config["seed"]\n    rng = np.random.default_rng(np.random.SeedSequence([seed, term, course_number]))\n    student_rng = np.random.default_rng(seed)\n    # Stable learner-level preparation across terms; none of these arrays leaves\n    # the generator or is placed in the source/feature contract.\n    prep = student_rng.normal(0, 1, size["students"])\n    base_engagement = student_rng.normal(0, .8, size["students"])\n    start = ORIGIN + (term - 1) * 120 * DAY\n    end = start + 84 * DAY\n    publication = start + (7 * config["current_week"] if term == 5 else 114) * DAY\n    created = start - 30 * DAY\n    cid = term * 1000 + course_number\n    rows = {name: [] for name in contract["tables"] if name not in STATIC}\n    versions = {}\n\n    def emit(table, row, event=None, recorded=None):\n        event = event if event is not None else created\n        recorded = recorded if recorded is not None else event + pd.Timedelta(hours=2)\n        key = table, row["id"]\n        versions[key] = versions.get(key, 0) + 1\n        rows[table].append(dict(row, effective_at=event, recorded_at=recorded, revision=versions[key]))\n\n    def dated(row, state="active", at=created):\n        return dict(row, created_at=at, updated_at=at, workflow_state=state)\n\n    course = dated({"id": cid, "name": f"Synthetic Course {course_number:03d}, Term {term}",\n                    "account_id": 1, "enrollment_term_id": term, "start_at": start,\n                    "conclude_at": end}, "available")\n    emit("courses", course)\n    for s in range(size["sections_per_course"]):\n        emit("course_sections", dated({"id": cid * 10 + s, "course_id": cid, "enrollment_term_id": term,\n             "name": f"Section {s + 1}", "start_at": start, "end_at": end}))\n    emit("assignment_groups", dated({"id": cid, "name": "Point-weighted coursework",\n                                    "context_id": cid, "context_type": "Course", "group_weight": 100.}, "available"))\n    assignments = []\n    for a in range(size["assignments_per_course"]):\n        aid = cid * 100 + a\n        due_day = 4 + (77 * a / max(1, size["assignments_per_course"] - 1))\n        mode = "on_paper" if a % 9 == 3 else ("external_tool" if a % 11 == 5 else "online_upload")\n        assignment = dated({"id": aid, "title": f"Synthetic assessment {a + 1}",\n                            "context_id": cid, "context_type": "Course", "assignment_group_id": cid,\n                            "due_at": start + due_day * DAY, "unlock_at": start + max(0, due_day - 14) * DAY,\n                            "points_possible": float(rng.choice([50, 100, 150])),\n                            "submission_types": [mode], "grading_type": "points", "omit_from_final_grade": False}, "published")\n        emit("assignments", assignment)\n        assignments.append(assignment)\n    teacher_id = size["students"] + course_number\n    teacher_enrollment = dated({"id": cid * 1000 + 999, "user_id": teacher_id, "course_id": cid,\n                               "course_section_id": cid * 10, "type": "TeacherEnrollment",\n                               "start_at": start, "end_at": end})\n    emit("enrollments", teacher_enrollment)\n    pool = int(size["students"] * .8) if term <= 2 else size["students"]\n    learner_ids = rng.choice(np.arange(1, pool + 1), size["enrollments_per_course"], replace=False)\n    difficulty = rng.normal(0, .5)\n    withdrawals, qualifications = [], []\n    event_serial = 0\n\n    def log_event(uid, when):\n        nonlocal event_serial\n        event_serial += 1\n        # A deterministic UUID per unique course request, not a learner key.\n        log_id = str(uuid.UUID(bytes=hashlib.sha256(f"{seed}:{cid}:{event_serial}".encode()).digest()[:16]))\n        row = {"id": log_id, "timestamp": when, "user_id": uid, "course_id": cid,\n               "url": f"/courses/{cid}/assignments", "http_method": "GET", "http_status": 200,\n               "http_version": "2.0", "remote_ip": f"192.0.2.{1 + event_serial % 253}",\n               "interaction_micros": int(rng.integers(1000, 90000)), "participated": bool(rng.random() < .35)}\n        emit("web_logs", row, when, when + rng.uniform(.02, 36) * pd.Timedelta(hours=1))\n\n    for i, uid in enumerate(learner_ids):\n        uid = int(uid)\n        eid = cid * 1000 + i\n        enrollment_day = int(rng.choice([0, 0, 0, 0, 0, 5, 12]))\n        enrolled = start + enrollment_day * DAY\n        enrollment = dated({"id": eid, "user_id": uid, "course_id": cid,\n                             "course_section_id": cid * 10 + i % size["sections_per_course"],\n                             "type": "StudentEnrollment", "start_at": enrolled, "end_at": None},\n                            at=enrolled - DAY)\n        emit("enrollments", enrollment, enrolled - DAY)\n        shock_week = int(rng.integers(2, 10))\n        shock = rng.exponential(.85) if rng.random() < .4 else 0.\n        recovery = rng.uniform(.08, .45)\n        innovation = rng.normal(0, .24, 12)\n        engagement = np.empty(12)\n        performance = np.empty(12)\n        withdrawn_at = None\n        for w in range(12):\n            disruption = shock * np.exp(-recovery * max(0, w - shock_week)) if w >= shock_week else 0.\n            engagement[w] = .9 * base_engagement[uid - 1] - disruption + innovation[w] + .02 * w\n            performance[w] = 85 + 14 * prep[uid - 1] - 9 * difficulty + 4 * engagement[w] - 6 * disruption\n            hazard = .004 + .026 * _sigmoid(-engagement[w] - .5 * prep[uid - 1] + disruption - 1.1)\n            if withdrawn_at is None and w >= 1 and rng.random() < hazard:\n                withdrawn_at = start + (w * 7 + rng.uniform(0, 7)) * DAY\n            num_logs = int(rng.poisson(np.clip(3.8 + 2.1 * engagement[w], .15, 12)))\n            for _ in range(num_logs):\n                event = start + (w * 7 + rng.uniform(0, 7)) * DAY\n                if event >= enrolled and (withdrawn_at is None or event < withdrawn_at):\n                    log_event(uid, event)\n        if withdrawn_at is not None:\n            recorded = withdrawn_at + rng.uniform(.1, 8) * DAY\n            withdrawals.append({"id": eid, "enrollment_id": eid, "effective_at": withdrawn_at,\n                                "recorded_at": recorded, "event_type": "CONFIRMED_COURSE_WITHDRAWAL"})\n            emit("enrollments", dict(enrollment, updated_at=recorded, end_at=withdrawn_at, workflow_state="inactive"),\n                 withdrawn_at, recorded)\n\n        earned, possible, graded_possible = 0., 0., 0.\n        incomplete = bool(rng.random() < .035)\n        for a, assignment in enumerate(assignments):\n            aid = assignment["id"]\n            sid = eid * 100 + a\n            due = assignment["due_at"]\n            w = min(11, int((due - start) / DAY) // 7)\n            if due < enrolled:\n                excused = True\n            else:\n                excused = rng.random() < .025\n            paper = assignment["submission_types"][0] in ("on_paper", "external_tool")\n            submission = dated({"id": sid, "user_id": uid, "assignment_id": aid, "submitted_at": None,\n                                "graded_at": None, "score": None, "attempt": 0, "excused": False,\n                                "submission_type": None if paper else "online_upload"}, "unsubmitted", created)\n            emit("submissions", submission)\n            if rng.random() < .09:\n                override_time = due - rng.uniform(1, 5) * DAY\n                override = dated({"id": sid, "assignment_id": aid, "set_type": "ADHOC", "set_id": None,\n                                  "due_at": min(end - 2 * DAY, due + rng.uniform(2, 9) * DAY), "due_at_overridden": True,\n                                  "title": "Synthetic approved individual extension"}, at=override_time)\n                recorded = override_time + rng.uniform(.1, 4) * DAY\n                emit("assignment_overrides", override, override_time, recorded)\n                emit("assignment_override_students", dated({"id": sid, "user_id": uid, "assignment_id": aid,\n                     "assignment_override_id": sid}, at=override_time), override_time, recorded)\n                due = override["due_at"]\n            if excused:\n                at = max(enrolled, due - 3 * DAY)\n                emit("submissions", dict(submission, excused=True, updated_at=at), at, at + DAY)\n                continue\n            points = assignment["points_possible"]\n            possible += points\n            completion_probability = .57 + .425 * _sigmoid(1.5 * engagement[w] + .6 * prep[uid - 1] + 1.8)\n            completed = rng.random() < completion_probability and (withdrawn_at is None or due < withdrawn_at)\n            if not completed:\n                # Explicit instructor zero only after final reconciliation, not\n                # a conversion of a null in-progress submission into failure.\n                at = end + 7 * DAY\n                if not incomplete:\n                    emit("submissions", dict(submission, score=0., graded_at=at, updated_at=at, workflow_state="graded"),\n                         at, at + DAY)\n                    graded_possible += points\n                continue\n            delay = rng.normal(-.8, 1.8) + max(0, -engagement[w]) * 1.7\n            submitted = min(end + 2 * DAY, max(enrolled, due + delay * DAY))\n            grade_time = min(end + 12 * DAY, submitted + rng.uniform(1, 12) * DAY)\n            grade = float(np.clip(performance[w] + rng.normal(0, 8), 5, 100) * points / 100)\n            available_submission = dict(submission, submitted_at=None if paper else submitted,\n                                        attempt=1, updated_at=submitted, workflow_state="unsubmitted" if paper else "submitted")\n            if not paper:\n                emit("submissions", available_submission, submitted, submitted + rng.uniform(.05, 3) * DAY)\n            if not (incomplete and a >= len(assignments) - 3):\n                graded = dict(available_submission, score=grade, graded_at=grade_time,\n                              updated_at=grade_time, workflow_state="graded")\n                emit("submissions", graded, grade_time, grade_time + rng.uniform(.05, 2) * DAY)\n                graded_possible += points\n                if not paper and rng.random() < .1:\n                    retry = min(end + 14 * DAY, grade_time + rng.uniform(1, 4) * DAY)\n                    retry_grade = min(end + 18 * DAY, retry + rng.uniform(1, 4) * DAY)\n                    grade = float(np.clip(grade + rng.normal(4, 5) * points / 100, 0, points))\n                    emit("submissions", dict(available_submission, attempt=2, submitted_at=retry,\n                         updated_at=retry, workflow_state="submitted"), retry, retry + .1 * DAY)\n                    emit("submissions", dict(graded, score=grade, attempt=2, submitted_at=retry,\n                         graded_at=retry_grade, updated_at=retry_grade), retry_grade, retry_grade + .1 * DAY)\n                earned += grade\n        final_score = 100 * earned / possible if possible else None\n        current_score = 100 * earned / graded_possible if graded_possible else None\n        score_time = end + 20 * DAY\n        emit("scores", {"id": eid, "enrollment_id": eid, "course_score": True,\n                       "current_score": current_score, "final_score": final_score, "updated_at": score_time},\n             score_time, score_time + DAY)\n        qualifications.append({"enrollment_id": eid, "effective_at": end + 21 * DAY, "recorded_at": end + 22 * DAY,\n                               "qualified": bool(not incomplete and possible > 0),\n                               "reason": "final_reconciliation_complete" if not incomplete else "incomplete_gradebook",\n                               "course_finished_at": end})\n        if withdrawn_at is None:\n            emit("enrollments", dict(enrollment, updated_at=end, end_at=end, workflow_state="completed"), end, end + DAY)\n    emit("courses", dict(course, updated_at=end, workflow_state="completed"), end, end + DAY)\n    # Logs legitimately include requests without a local enrolled learner.\n    log_event(None, start + DAY)\n    log_event(size["students"] + size["courses_per_term"] + 100000, start + 2 * DAY)\n    revision_frames = {name: frame(name, records, contract, history=True) for name, records in rows.items()}\n    if term == 5:\n        revision_frames = {name: data.loc[(data.effective_at <= publication) & (data.recorded_at <= publication)].reset_index(drop=True)\n                           for name, data in revision_frames.items()}\n    wd = pd.DataFrame(withdrawals, columns=["id", "enrollment_id", "effective_at", "recorded_at", "event_type"])\n    quality = pd.DataFrame(qualifications)\n    for data in (wd, quality):\n        for name in ("effective_at", "recorded_at"):\n            data[name] = pd.to_datetime(data[name], utc=True).dt.floor("us")\n    if term == 5:\n        wd = wd.loc[(wd.effective_at <= publication) & (wd.recorded_at <= publication)].reset_index(drop=True)\n        quality = quality.iloc[:0].copy()\n    return CourseBatch(term, cid, start, end, publication, revision_frames, wd, quality)\n\n\ndef validate_batch(batch, dims, contract):\n    sources = {name: batch.source(name).drop(columns=list(HISTORY)) for name in batch.revisions}\n    references = dict(dims, **sources)\n    counts = {}\n    for name, data in batch.revisions.items():\n        validate_table(name, data, contract, references, history=True)\n        counts[name] = validate_table(name, sources[name], contract, references)\n    enrollments = sources["enrollments"]\n    if enrollments.duplicated(["user_id", "course_id", "type"]).any():\n        raise ValueError("Duplicate learner-course enrollment")\n    sections = sources["course_sections"].set_index("id")\n    if not enrollments.course_section_id.map(sections.course_id).eq(enrollments.course_id).all():\n        raise ValueError("Enrollment section belongs to a different course")\n    assignments = sources["assignments"].set_index("id")\n    sub = batch.revisions["submissions"]\n    if not sub.user_id.isin(enrollments.user_id).all():\n        raise ValueError("Submission user is not enrolled")\n    if (sub.score.dropna() < 0).any() or (sub.score > sub.assignment_id.map(assignments.points_possible)).any():\n        raise ValueError("Submission score outside point range")\n    if ((sub.graded_at < sub.submitted_at) & sub.graded_at.notna() & sub.submitted_at.notna()).any():\n        raise ValueError("Grade precedes submission")\n    if not batch.withdrawal_events.enrollment_id.isin(enrollments.id).all():\n        raise ValueError("Orphan supplemental withdrawal")\n    if batch.withdrawal_events.duplicated("enrollment_id").any():\n        raise ValueError("Duplicate confirmed withdrawal")\n    if batch.term == 5:\n        if len(batch.qualification) or len(sources["scores"]):\n            raise ValueError("Current-term future label contamination")\n        if any((d.recorded_at > batch.publication).any() or (d.effective_at > batch.publication).any()\n               for d in batch.revisions.values()):\n            raise ValueError("Current-term future source contamination")\n    return counts\n', 'aews.core.features': 'import numpy as np\nimport pandas as pd\n\nfrom aews.core.simulation import DAY\n\nFEATURE_VERSION = "past-only-weekly-1"\nFEATURES = [\n    "days_enrolled", "due_count", "completion_ratio", "missing_online_ratio",\n    "late_ratio", "paper_unknown_ratio", "excused_ratio", "graded_ratio",\n    "grade_mean", "grade_recent_mean", "grade_trend", "reattempt_ratio",\n    "log_count", "recent_log_count", "activity_change", "days_since_activity",\n]\nCLASSES = ["excellent", "fail", "pass", "withdrawn"]\nPARTITIONS = {1: "train", 2: "train", 3: "validation", 4: "previously_exposed_diagnostic", 5: "score"}\n\n\ndef ratio(numerator, denominator):\n    return float(numerator / denominator) if denominator else np.nan\n\n\ndef weekly_features(batch, week):\n    cutoff = batch.start + 7 * week * DAY\n    if batch.term == 5 and cutoff > batch.publication:\n        raise ValueError("Cannot score an unpublished future cutoff")\n    e = batch.source("enrollments", cutoff)\n    e = e.loc[(e.type == "StudentEnrollment") & (e.workflow_state == "active") &\n              (e.start_at.isna() | (e.start_at <= cutoff))].copy()\n    known_wd = batch.withdrawal_events\n    known_wd = known_wd.loc[(known_wd.effective_at <= cutoff) & (known_wd.recorded_at <= cutoff)]\n    e = e.loc[~e.id.isin(known_wd.enrollment_id)]\n    a = batch.source("assignments", cutoff)\n    a = a.loc[(a.context_type == "Course") & (a.context_id == batch.course_id) &\n              (a.workflow_state == "published") & (~a.omit_from_final_grade) &\n              (a.unlock_at.isna() | (a.unlock_at <= cutoff))]\n    a = a.rename(columns={"id": "assignment_id"})\n    s = batch.source("submissions", cutoff)\n    s = s[["user_id", "assignment_id", "submitted_at", "graded_at", "score", "attempt", "excused"]]\n    overrides = batch.source("assignment_overrides", cutoff)\n    mapping = batch.source("assignment_override_students", cutoff)\n    overrides = overrides.loc[(overrides.workflow_state == "active") & (overrides.set_type == "ADHOC") &\n                              overrides.due_at_overridden]\n    mapping = mapping.loc[mapping.workflow_state == "active"]\n    extension = mapping[["user_id", "assignment_id", "assignment_override_id"]].merge(\n        overrides[["id", "due_at"]], left_on="assignment_override_id", right_on="id", how="inner")\n    if extension.duplicated(["user_id", "assignment_id"]).any():\n        raise ValueError("Ambiguous overlapping individual overrides")\n    extension = extension[["user_id", "assignment_id", "due_at"]].rename(columns={"due_at": "override_due"})\n    work = e[["id", "user_id", "start_at"]].merge(\n        a[["assignment_id", "due_at", "points_possible", "submission_types"]], how="cross")\n    work = work.merge(s, on=["user_id", "assignment_id"], how="left", validate="one_to_one")\n    work = work.merge(extension, on=["user_id", "assignment_id"], how="left", validate="one_to_one")\n    work["effective_due"] = work.override_due.combine_first(work.due_at)\n    work["due"] = (work.effective_due <= cutoff) & (work.effective_due >= work.start_at)\n    work["excused"] = work.excused.fillna(False).astype(bool)\n    work["paper"] = work.submission_types.map(lambda x: bool(set(x) & {"on_paper", "external_tool"}))\n    work["submitted"] = work.submitted_at.notna() & (work.submitted_at <= cutoff)\n    work["graded"] = work.graded_at.notna() & (work.graded_at <= cutoff) & work.score.notna()\n    work["normalized_grade"] = work.score * 100 / work.points_possible\n    logs = batch.source("web_logs", cutoff)\n    logs = logs.loc[(logs.timestamp >= batch.start) & (logs.timestamp <= cutoff) & (logs.course_id == batch.course_id)]\n    rows = []\n    work_groups = dict(tuple(work.groupby("id", sort=False)))\n    log_groups = dict(tuple(logs.groupby("user_id", sort=False)))\n    for enrollment in e.itertuples(index=False):\n        items = work_groups.get(enrollment.id, work.iloc[:0])\n        due_all = items.loc[items.due]\n        due = due_all.loc[~due_all.excused]\n        online = due.loc[~due.paper]\n        known_grade = due.loc[due.graded & (due.points_possible > 0)]\n        recent_grade = known_grade.loc[known_grade.graded_at > cutoff - 14 * DAY]\n        prior_grade = known_grade.loc[known_grade.graded_at <= cutoff - 14 * DAY]\n        observed_logs = log_groups.get(enrollment.user_id, logs.iloc[:0])\n        observed_logs = observed_logs.loc[observed_logs.timestamp >= enrollment.start_at]\n        recent_count = int((observed_logs.timestamp > cutoff - 14 * DAY).sum())\n        prior_count = int(((observed_logs.timestamp > cutoff - 28 * DAY) &\n                           (observed_logs.timestamp <= cutoff - 14 * DAY)).sum())\n        complete = due.submitted | (due.paper & due.graded & due.score.gt(0))\n        last_log = observed_logs.timestamp.max()\n        window_start = max(batch.start, enrollment.start_at)\n        row = {\n            "enrollment_id": int(enrollment.id), "user_id": int(enrollment.user_id),\n            "course_id": batch.course_id, "term": batch.term, "cutoff_week": week,\n            "as_of_time": cutoff, "observation_start": window_start, "observation_end": cutoff,\n            "feature_version": FEATURE_VERSION, "partition": PARTITIONS[batch.term],\n            "days_enrolled": float((cutoff - window_start) / DAY),\n            "due_count": float(len(due)), "completion_ratio": ratio(complete.sum(), len(due)),\n            "missing_online_ratio": ratio((~online.submitted).sum(), len(online)),\n            "late_ratio": ratio((online.submitted & (online.submitted_at > online.effective_due)).sum(),\n                                online.submitted.sum()),\n            "paper_unknown_ratio": ratio((due.paper & ~due.graded).sum(), len(due)),\n            "excused_ratio": ratio(due_all.excused.sum(), len(due_all)),\n            "graded_ratio": ratio(len(known_grade), len(due)),\n            "grade_mean": ratio(known_grade.score.sum() * 100, known_grade.points_possible.sum()),\n            "grade_recent_mean": ratio(recent_grade.score.sum() * 100, recent_grade.points_possible.sum()),\n            "grade_trend": (ratio(recent_grade.score.sum() * 100, recent_grade.points_possible.sum()) -\n                            ratio(prior_grade.score.sum() * 100, prior_grade.points_possible.sum())),\n            "reattempt_ratio": ratio(due.attempt.gt(1).sum(), len(due)),\n            "log_count": float(len(observed_logs)), "recent_log_count": float(recent_count),\n            "activity_change": float(recent_count - prior_count),\n            "days_since_activity": float((cutoff - last_log) / DAY) if pd.notna(last_log) else float((cutoff - window_start) / DAY + 1),\n            "data_quality_flag": "no_due_work" if not len(due) else ("no_known_grades" if not len(known_grade) else "usable"),\n        }\n        rows.append(row)\n    return pd.DataFrame(rows)\n\n\ndef labels(batch, cutoff):\n    """Read outcome-side future only for finalized historical terms."""\n    if batch.term == 5:\n        raise ValueError("Term-five hidden truth is never exposed to labels")\n    enrollments = batch.source("enrollments")\n    scores = batch.source("scores")\n    scores = scores.loc[scores.course_score]\n    if scores.duplicated("enrollment_id").any():\n        raise ValueError("Multiple aggregate course scores for one enrollment")\n    score_map = scores.set_index("enrollment_id").final_score.to_dict()\n    q = batch.qualification\n    q = q.loc[(q.recorded_at <= batch.publication) & (q.effective_at <= batch.publication)]\n    quality = q.set_index("enrollment_id").to_dict("index")\n    wd = batch.withdrawal_events\n    wd = wd.loc[(wd.recorded_at <= batch.publication) & (wd.effective_at <= batch.end)]\n    wd_map = wd.set_index("enrollment_id").effective_at.to_dict()\n    result = []\n    for e in enrollments.loc[enrollments.type == "StudentEnrollment"].itertuples(index=False):\n        outcome, reason = None, None\n        effective = wd_map.get(e.id)\n        if effective is not None:\n            if effective <= cutoff:\n                reason = "withdrawn_before_cutoff_including_late_recording"\n            else:\n                outcome = "withdrawn"\n        elif e.id not in quality or not quality[e.id]["qualified"]:\n            reason = "incomplete_or_unqualified_gradebook"\n        elif quality[e.id]["course_finished_at"] > batch.publication:\n            reason = "course_not_finished"\n        elif pd.isna(score_map.get(e.id)):\n            reason = "unknown_final_grade"\n        else:\n            grade = score_map[e.id]\n            outcome = "fail" if grade < 60 else ("excellent" if grade >= 90 else "pass")\n        result.append({"enrollment_id": int(e.id), "label": outcome, "label_status": reason or "known",\n                       "label_available_at": batch.publication})\n    return pd.DataFrame(result)\n\n\ndef feature_contract():\n    return {\n        "version": FEATURE_VERSION, "predictors": FEATURES,\n        "units": {"grade_mean": "percent", "grade_recent_mean": "percent", "grade_trend": "percentage_point_difference",\n                  "days_enrolled": "days", "days_since_activity": "days", "activity_change": "request_count_difference"},\n        "entity": "enrollment_id + cutoff_week; course weeks relative to course start",\n        "availability": "Both effective_at and recorded_at <= cutoff; no latest-state historical filtering",\n        "missing_values": "NaN denotes unknown or zero eligible denominator, train-only median imputation plus missing indicators",\n        "deadline": "Known active individual extension supersedes known base due_at; only due work after enrollment start",\n        "paper": "Ungraded paper/external work is unknown, never inferred missing from null submitted_at",\n        "prohibited_predictors": ["names", "identifiers", "protected_traits", "latent_state", "final_score", "future_withdrawal", "label"],\n        "current_truth": "Not exposed or persisted; no current labels",\n        "partitions": PARTITIONS, "classes": CLASSES,\n        "warning": "Four course outcomes; synthetic correlations do not demonstrate institutional validity or causality",\n    }\n', 'aews.core.execution': 'import hashlib\nimport json\nimport platform\nfrom collections import Counter\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport pyarrow as pa\nimport pyarrow.parquet as pq\nimport sklearn\n\nfrom aews.core.contract import HISTORY, dumps, validate_table\nfrom aews.core.features import FEATURE_VERSION, FEATURES, feature_contract, labels, weekly_features\nfrom aews.core.simulation import DAY, STATIC, dimensions, generate_course, validate_batch, validate_config\n\n\ndef arrow_schema(table, contract, history=False):\n    mapping = {"int64": pa.int64(), "int32": pa.int32(), "float64": pa.float64(), "bool": pa.bool_(),\n               "datetime": pa.timestamp("us", tz="UTC"), "string": pa.string(), "array<string>": pa.list_(pa.string())}\n    fields = [pa.field(name, mapping[spec["type"]], nullable=spec["nullable"])\n              for name, spec in contract["tables"][table]["columns"].items()]\n    if history:\n        fields += [pa.field("effective_at", pa.timestamp("us", tz="UTC"), nullable=False),\n                   pa.field("recorded_at", pa.timestamp("us", tz="UTC"), nullable=False),\n                   pa.field("revision", pa.int32(), nullable=False)]\n    return pa.schema(fields)\n\n\nSUPPORT_SCHEMA = {\n    "course_withdrawal_events": pa.schema([\n        ("id", pa.int64()), ("enrollment_id", pa.int64()), ("effective_at", pa.timestamp("us", tz="UTC")),\n        ("recorded_at", pa.timestamp("us", tz="UTC")), ("event_type", pa.string())]),\n    "gradebook_qualification": pa.schema([\n        ("enrollment_id", pa.int64()), ("effective_at", pa.timestamp("us", tz="UTC")),\n        ("recorded_at", pa.timestamp("us", tz="UTC")), ("qualified", pa.bool_()), ("reason", pa.string()),\n        ("course_finished_at", pa.timestamp("us", tz="UTC"))]),\n}\n\n\nclass ParquetSourceSink:\n    """One bounded course at a time; at most one writer per table per term."""\n    def __init__(self, root, contract):\n        self.root, self.contract = Path(root), contract\n        self.writers = {}\n        self.term = None\n\n    def write(self, namespace, table, data, schema):\n        key = namespace, table\n        path = self.root / namespace / table\n        path.mkdir(parents=True, exist_ok=True)\n        if key not in self.writers:\n            self.writers[key] = pq.ParquetWriter(str(path / f"part-{self.term or 0:02d}.parquet"), schema, compression="snappy")\n        arrow = pa.Table.from_pandas(data, schema=schema, preserve_index=False, safe=True)\n        arrow.validate(full=True)\n        self.writers[key].write_table(arrow, row_group_size=25000)\n\n    def dimensions(self, dims):\n        for table, data in dims.items():\n            self.write("canvas", table, data, arrow_schema(table, self.contract))\n        self.close()\n\n    def batch(self, batch):\n        if batch.term != self.term:\n            self.close()\n            self.term = batch.term\n        for table, data in batch.revisions.items():\n            namespace = self.contract["tables"][table]["namespace"]\n            self.write(namespace, table, batch.source(table).drop(columns=list(HISTORY)), arrow_schema(table, self.contract))\n            self.write("sim_history", table + "_revisions", data, arrow_schema(table, self.contract, history=True))\n        self.write("sim_support", "course_withdrawal_events", batch.withdrawal_events, SUPPORT_SCHEMA["course_withdrawal_events"])\n        self.write("sim_support", "gradebook_qualification", batch.qualification, SUPPORT_SCHEMA["gradebook_qualification"])\n\n    def close(self):\n        for writer in self.writers.values():\n            writer.close()\n        self.writers = {}\n\n\ndef environment_versions():\n    return {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,\n            "scikit_learn": sklearn.__version__, "pyarrow": pa.__version__}\n\n\ndef prepare(config, contract, sink=None):\n    size = validate_config(config)\n    dims = dimensions(config, contract)\n    for name, data in dims.items():\n        validate_table(name, data, contract, dims)\n    if sink is not None:\n        sink.dimensions(dims)\n    counts = Counter({k: len(v) for k, v in dims.items()})\n    revisions = Counter()\n    scenario_counts = Counter()\n    frames, excluded_frames = [], []\n    prior_max = {}\n    seen_courses = set()\n    try:\n        for term in range(1, 6):\n            term_counts = Counter()\n            for course_number in range(1, size["courses_per_term"] + 1):\n                batch = generate_course(config, contract, term, course_number)\n                if batch.course_id in seen_courses:\n                    raise ValueError("Duplicate course chunk")\n                seen_courses.add(batch.course_id)\n                batch_counts = validate_batch(batch, dims, contract)\n                for table, amount in batch_counts.items():\n                    counts[table] += amount\n                    term_counts[table] += amount\n                    history = batch.revisions[table]\n                    revisions[table] += len(history)\n                    # Disjoint monotone integer key ranges validate cross-chunk\n                    # PK uniqueness without an unbounded Python set of IDs.\n                    if table != "web_logs" and len(history):\n                        lower, upper = int(history.id.min()), int(history.id.max())\n                        if lower <= prior_max.get(table, -1):\n                            raise ValueError(f"Overlapping cross-course PK range: {table}")\n                        prior_max[table] = upper\n                for week in config["cutoff_weeks"]:\n                    if term == 5 and week > config["current_week"]:\n                        continue\n                    features = weekly_features(batch, week)\n                    if term < 5:\n                        outcome = labels(batch, batch.start + 7 * week * DAY)\n                        features = features.merge(outcome, on="enrollment_id", how="left", validate="one_to_one")\n                        excluded = features.loc[features.label.isna(), ["enrollment_id", "term", "cutoff_week", "label_status"]]\n                        excluded_frames.append(excluded)\n                    else:\n                        # Current silver table will not contain label columns.\n                        features["label"] = None\n                        features["label_status"] = "current_term_no_truth_published"\n                        features["label_available_at"] = pd.NaT\n                    frames.append(features)\n                logs = batch.source("web_logs")\n                scenario_counts["anonymous_web_requests"] += int(logs.user_id.isna().sum())\n                scenario_counts["external_web_requests"] += int((logs.user_id.notna() & ~logs.user_id.isin(dims["users"].id)).sum())\n                sub = batch.source("submissions")\n                scenario_counts["reattempted_submissions"] += int(sub.attempt.gt(1).sum())\n                scenario_counts["excused_submissions"] += int(sub.excused.fillna(False).sum())\n                scenario_counts["graded_without_submitted_timestamp"] += int((sub.graded_at.notna() & sub.submitted_at.isna() & sub.score.gt(0)).sum())\n                scenario_counts["confirmed_withdrawals"] += len(batch.withdrawal_events)\n                scenario_counts["individual_extensions"] += len(batch.source("assignment_overrides"))\n                if term < 5:\n                    scores = batch.source("scores")\n                    withdrawn = set(batch.withdrawal_events.enrollment_id)\n                    scenario_counts["passing_grade_withdrawals"] += int((scores.enrollment_id.isin(withdrawn) & scores.final_score.ge(60)).sum())\n                    scenario_counts["failing_without_withdrawal"] += int((~scores.enrollment_id.isin(withdrawn) & scores.final_score.lt(60)).sum())\n                    scenario_counts["unqualified_gradebooks"] += int((~batch.qualification.qualified).sum())\n                if sink is not None:\n                    sink.batch(batch)\n            print(dumps({"term": term, "source_counts": term_counts}), flush=True)\n    finally:\n        if sink is not None:\n            sink.close()\n    features = pd.concat(frames, ignore_index=True)\n    features["label_available_at"] = pd.to_datetime(features.label_available_at, utc=True)\n    if features.duplicated(["enrollment_id", "cutoff_week"]).any():\n        raise ValueError("Duplicate feature snapshot key")\n    if features.groupby("enrollment_id").partition.nunique().gt(1).any():\n        raise ValueError("Temporal split crosses enrollment")\n    if features.loc[features.term == 5, "label"].notna().any():\n        raise ValueError("Current-term label contamination")\n    if not all(scenario_counts[name] > 0 for name in (\n        "confirmed_withdrawals", "passing_grade_withdrawals", "failing_without_withdrawal", "unqualified_gradebooks")):\n        raise ValueError(f"Required stochastic counterexample absent; do not silently regenerate: {scenario_counts}")\n    excluded = pd.concat(excluded_frames, ignore_index=True)\n    target = {\n        "student_users": size["students"], "teacher_users": size["courses_per_term"],\n        "courses": 5 * size["courses_per_term"], "sections": 5 * size["courses_per_term"] * size["sections_per_course"],\n        "student_enrollments": 5 * size["courses_per_term"] * size["enrollments_per_course"],\n        "submission_records": 5 * size["courses_per_term"] * size["enrollments_per_course"] * size["assignments_per_course"],\n        "full_profile_web_log_target": "2-3 million full longitudinal events; published current term stops at cutoff, so published count is lower",\n    }\n    summary = {\n        "synthetic": True, "profile": config["mode"], "seed": config["seed"], "target_counts": target,\n        "observed_source_counts": dict(counts), "observed_history_counts": dict(revisions),\n        "observed_scenarios": dict(scenario_counts), "feature_snapshots": len(features),\n        "label_counts_by_term": features.groupby(["term", "label"], dropna=False).size().rename("rows").reset_index().to_dict("records"),\n        "excluded_label_counts": excluded.label_status.value_counts().to_dict(),\n        "assertions": ["all_selected_source_types_nullability_enums_checked", "no_duplicate_source_PK",\n                       "cross_course_integer_PK_ranges_disjoint", "web_UUIDs_derived_from_unique_course_and_request",\n                       "mandatory_projected_FK_integrity", "section_course_consistency", "grade_point_bounds",\n                       "source_revision_both_times_checked", "current_future_truth_not_published",\n                       "snapshot_keys_unique", "enrollment_temporal_partitions_disjoint"],\n        "environment": environment_versions(),\n        "limitations": [\n            "Selected source projection, not complete Canvas replication.",\n            "Historical web data is synthetic; production DAP retention is 30 days.",\n            "All term-four course offering IDs are new by construction; this is not evidence of unseen curriculum generalization.",\n            "Returning learners cross historical terms; the temporal test is not entirely learner-disjoint.",\n            "Unknown gradebooks remain unlabeled, rather than becoming fail.",\n            "No causal or real-world validity claim; all performance describes synthetic data.",\n            "Week-four model only. Optional week-two/six snapshots are not pooled into training or represented as validated deployments.",\n        ],\n    }\n    return features, summary\n\n\ndef evaluation_frame(result):\n    return pd.DataFrame([{\n        "model_version": result.version, "model_name": row["model_name"], "split": row["split"],\n        "slice": row["slice"], "metrics_json": dumps(row["metrics"]), "synthetic": True,\n        "selected": row["model_name"] == result.model_name,\n    } for row in result.evaluations])\n\n\ndef quality_frame(summary):\n    return pd.DataFrame([{"category": key, "details_json": dumps(value), "synthetic": True}\n                         for key, value in summary.items()])\n\n', 'aews.configuration': 'import json\nimport re\nfrom pathlib import Path\n\nfrom aews.core.simulation import validate_config\n\n\ndef load_configuration(directory):\n    directory = Path(directory)\n    values = [json.loads((directory / name).read_text(encoding="utf-8"))\n              for name in ("simulation.json", "modeling.json", "canvas-source-contract.json")]\n    validate_configuration(*values)\n    return tuple(values)\n\n\ndef validate_configuration(source, model, contract):\n    validate_config(source)\n    if source["mode"] not in ("small", "full") or source["current_week"] != 4:\n        raise ValueError("This starter supports small/full profiles and current week four only.")\n    for field in ("intended_workspace", "intended_lakehouse", "experiment_name"):\n        value = source.get(field)\n        if not isinstance(value, str) or not value.strip() or any(c in value for c in "`\\r\\n/\\\\"):\n            raise ValueError(f"Missing or unsafe configuration name: {field}")\n    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", source["intended_lakehouse"]):\n        raise ValueError("Use an alphanumeric/underscore Lakehouse name.")\n    if model.get("registration_approved") is not False:\n        raise ValueError("Accepted-model registration is not implemented.")\n    if model["workflow_version"] != "four-outcome-calibrated-1":\n        raise ValueError("Unsupported model workflow.")\n    if model["calibration_method"] != "sigmoid" or not 0.2 <= model["calibration_fraction_of_term3"] <= 0.8:\n        raise ValueError("Unsupported calibration protocol.")\n    if not model["logistic_C"] or any(c <= 0 for c in model["logistic_C"]):\n        raise ValueError("Logistic regularization values must be positive.")\n    if not model["class_weight_candidates"] or any(c not in (None, "balanced") for c in model["class_weight_candidates"]):\n        raise ValueError("Unsupported class weights.")\n    if model["max_training_rows"] > 100000 or model["max_training_rows"] <= 0:\n        raise ValueError("Model driver bound must be between one and 100000.")\n    if set(item["namespace"] for item in contract["tables"].values()) != {"canvas", "canvas_logs"}:\n        raise ValueError("Unexpected source namespaces.")\n', 'aews.parquet': '"""Parquet serialization compatible with the Fabric/Spark timestamp reader."""\nimport json\n\nimport pandas as pd\nimport pyarrow as pa\nimport pyarrow.parquet as pq\n\n\ndef write_frame(data, path):\n    table = pa.Table.from_pandas(data, preserve_index=False)\n    timestamps = {field.name for field in table.schema if pa.types.is_timestamp(field.type)}\n    schema = pa.schema([\n        field.with_type(pa.timestamp("us", tz=field.type.tz)) if field.name in timestamps else field\n        for field in table.schema\n    ], metadata=table.schema.metadata)\n    # Writer coercion alone leaves the embedded Arrow schema at nanoseconds.\n    # A safe cast changes that schema too, rejecting any lossy conversion.\n    table = table.cast(schema, safe=True)\n    metadata = dict(table.schema.metadata or {})\n    if b"pandas" in metadata:\n        pandas_metadata = json.loads(metadata[b"pandas"])\n        for column in pandas_metadata["columns"]:\n            if column["field_name"] in timestamps:\n                column["numpy_type"] = "datetime64[us]"\n        metadata[b"pandas"] = json.dumps(pandas_metadata).encode("utf-8")\n    table = table.replace_schema_metadata(metadata)\n    pq.write_table(table, path, version="1.0", coerce_timestamps="us",\n                   allow_truncated_timestamps=False)\n\n\ndef read_frame(path):\n    return normalize_timestamp_units(pd.read_parquet(path, engine="pyarrow"))\n\n\ndef normalize_timestamp_units(data):\n    data = data.copy()\n    for column in data:\n        if pd.api.types.is_datetime64_any_dtype(data[column].dtype):\n            data[column] = data[column].dt.as_unit("ns")\n    return data\n', 'aews.data_bundle': '"""Verify and load the included synthetic dataset without generating new records."""\nfrom dataclasses import dataclass\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path, PurePosixPath\nimport re\nimport shutil\nimport stat\nimport zipfile\n\nimport numpy as np\nimport pandas as pd\nimport pyarrow as pa\nimport pyarrow.parquet as pq\n\nfrom aews.core.contract import dumps, frame, validate_table\nfrom aews.core.execution import SUPPORT_SCHEMA, arrow_schema\nfrom aews.core.features import CLASSES, FEATURES, FEATURE_VERSION, PARTITIONS\nfrom aews.parquet import normalize_timestamp_units\n\nFILENAME = "academic-early-warning-simulated-data.zip"\nPURPOSE = "academic_early_warning_synthetic_input"\nMAX_COMPRESSED = 100 * 1024 * 1024\nMAX_UNCOMPRESSED = 512 * 1024 * 1024\nMAX_MEMBER = 128 * 1024 * 1024\nMAX_FILES = 512\nFEATURE_METADATA = [\n    "enrollment_id", "user_id", "course_id", "term", "cutoff_week", "as_of_time",\n    "observation_start", "observation_end", "feature_version", "partition", "data_quality_flag",\n]\nLABEL_FIELDS = ["label", "label_status", "label_available_at"]\n\n\ndef sha256_file(path):\n    with Path(path).open("rb") as handle:\n        return hashlib.file_digest(handle, "sha256").hexdigest() if hasattr(hashlib, "file_digest") else _hash_stream(handle)\n\n\ndef _hash_stream(handle):\n    digest = hashlib.sha256()\n    for chunk in iter(lambda: handle.read(1024 * 1024), b""):\n        digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef generation_settings(config):\n    return {key: config[key] for key in ("seed", "terms", "term_days", "cutoff_weeks", "current_week")} | {\n        "profile": "small", "size": config["profiles"]["small"],\n    }\n\n\ndef source_schemas(contract):\n    result = {}\n    for table, definition in contract["tables"].items():\n        result[definition["namespace"] + "." + table] = arrow_schema(table, contract)\n        if table not in ("accounts", "users", "enrollment_terms"):\n            result["sim_history." + table + "_revisions"] = arrow_schema(table, contract, history=True)\n    result.update({"sim_support." + name: schema for name, schema in SUPPORT_SCHEMA.items()})\n    return result\n\n\ndef safe_member_name(name):\n    path = PurePosixPath(name)\n    if (not name or name.startswith("/") or "\\\\" in name or ":" in name or\n            any(part in ("", ".", "..") for part in name.split("/")) or path.is_absolute()):\n        raise ValueError("Unsafe dataset archive path.")\n    return name\n\n\nclass _DirectoryArchive:\n    """Expose a bounded extracted dataset through the same reader as a ZIP."""\n\n    def __init__(self, root):\n        self.root = Path(root)\n        if not self.root.is_dir() or self.root.is_symlink():\n            raise ValueError("The dataset must be a real extracted folder.")\n        self.entries = {}\n        for path in self.root.rglob("*"):\n            if path.is_symlink():\n                raise ValueError("Symbolic links are not allowed in the dataset folder.")\n            if path.is_dir():\n                continue\n            if not path.is_file():\n                raise ValueError("Unexpected non-file dataset entry.")\n            name = safe_member_name(path.relative_to(self.root).as_posix())\n            entry = zipfile.ZipInfo(name)\n            entry.file_size = path.stat().st_size\n            self.entries[name] = entry\n            if len(self.entries) > MAX_FILES:\n                raise ValueError("Too many files in the dataset folder.")\n\n    def __enter__(self):\n        return self\n\n    def __exit__(self, *args):\n        return False\n\n    def infolist(self):\n        return list(self.entries.values())\n\n    def namelist(self):\n        return list(self.entries)\n\n    def getinfo(self, name):\n        return self.entries[safe_member_name(name)]\n\n    def open(self, name):\n        entry = self.getinfo(name)\n        path = self.root\n        for part in ("", *PurePosixPath(name).parts):\n            path = path / part\n            if path.is_symlink():\n                raise ValueError("Dataset path changed to a symbolic link.")\n        if not path.resolve().is_relative_to(self.root.resolve()) or path.stat().st_size != entry.file_size:\n            raise ValueError("Dataset file changed after folder inspection.")\n        return path.open("rb")\n\n    def read(self, name):\n        if self.getinfo(name).file_size > MAX_MEMBER:\n            raise ValueError("Dataset member exceeds the uncompressed-size limit.")\n        with self.open(name) as handle:\n            raw = handle.read(MAX_MEMBER + 1)\n        if len(raw) != self.getinfo(name).file_size:\n            raise ValueError("Dataset file size changed while reading.")\n        return raw\n\n\ndef load_lakehouse_input(root, input_mode, input_path, info, config, contract):\n    if input_mode not in ("uploaded", "folder"):\n        raise ValueError("Choose uploaded for a ZIP or folder for an extracted dataset.")\n    if not isinstance(input_path, str) or not input_path.startswith("Files/"):\n        raise ValueError("The input path must start with Files/ in the default Lakehouse.")\n    safe_member_name(input_path)\n    root = Path(root).resolve()\n    path = root.joinpath(*PurePosixPath(input_path).parts[1:])\n    if not path.resolve().is_relative_to(root) or path.resolve() == root:\n        raise ValueError("Dataset path must be a child of the default Lakehouse Files folder.")\n    if input_mode == "uploaded" and input_path != info["expected_lakehouse_path"]:\n        raise ValueError("Upload the included archive to the exact documented Files path.")\n    if input_mode == "folder" and not path.is_dir():\n        raise FileNotFoundError("Upload the extracted dataset folder and set SOURCE_FOLDER to its Files path.")\n    if input_mode == "uploaded" and not path.is_file():\n        raise FileNotFoundError("Upload the included data ZIP to Files/uploads first.")\n    return load_bundle(path, info, config, contract)\n\n\ndef validate_feature_frames(historical, current):\n    for name, data, labels in (("historical", historical, LABEL_FIELDS), ("current", current, [])):\n        if set(data.columns) != set(FEATURES + FEATURE_METADATA + labels):\n            raise ValueError(f"Unexpected {name} feature columns, including prohibited future-label fields.")\n        if data.empty or len(data) > 100000:\n            raise ValueError("Feature population is empty or exceeds the bounded reader.")\n        if data.duplicated(["enrollment_id", "cutoff_week"]).any() or not data.cutoff_week.eq(4).all():\n            raise ValueError("Duplicate enrollment features or unsupported observation cutoff.")\n        if not data.feature_version.eq(FEATURE_VERSION).all():\n            raise ValueError("Feature definition version does not match this code.")\n        if not data.partition.eq(data.term.map(PARTITIONS)).all():\n            raise ValueError("Temporal partition mismatch.")\n        if np.isinf(data[FEATURES].to_numpy(dtype=float)).any():\n            raise ValueError("Infinite feature value.")\n        if not data.observation_end.eq(data.as_of_time).all() or (data.observation_start > data.observation_end).any():\n            raise ValueError("Invalid observation interval.")\n    if not set(historical.term) <= {1, 2, 3, 4} or not current.term.eq(5).all():\n        raise ValueError("Historical/current term separation failed.")\n    if set(historical.enrollment_id) & set(current.enrollment_id):\n        raise ValueError("An enrollment crosses historical/current datasets.")\n    if not set(historical.label.dropna()) <= set(CLASSES):\n        raise ValueError("Unknown target outcome.")\n    for terms in ([1, 2], [3]):\n        if set(historical.loc[historical.term.isin(terms), "label"].dropna()) != set(CLASSES):\n            raise ValueError("Training or development labels lack an outcome class.")\n    known = historical.loc[historical.label.notna()]\n    if known.label_available_at.isna().any() or (known.label_available_at <= known.as_of_time).any():\n        raise ValueError("Historical labels must become known after the prediction cutoff.")\n\n\n@dataclass\nclass DatasetBundle:\n    archive: Path\n    manifest: dict\n    features: pd.DataFrame\n    summary: dict\n    tables: dict\n\n    def extract_sources(self, destination):\n        destination = Path(destination)\n        if destination.exists():\n            raise FileExistsError("Dataset extraction requires a fresh staging directory.")\n        folder = self.manifest["verified_input_kind"] == "folder"\n        if not folder and sha256_file(self.archive) != self.manifest["verified_zip_sha256"]:\n            raise ValueError("The uploaded archive changed after validation.")\n        with (_DirectoryArchive(self.archive) if folder else zipfile.ZipFile(self.archive)) as archive:\n            if hashlib.sha256(archive.read("manifest.json")).hexdigest() != self.manifest["verified_manifest_sha256"]:\n                raise ValueError("The input manifest changed after validation.")\n            destination.mkdir(parents=True)\n            for member in self.manifest["members"]:\n                name = member["path"]\n                if not name.startswith("source_parquet/"):\n                    continue\n                safe_member_name(name)\n                target = destination.joinpath(*PurePosixPath(name).parts[1:])\n                target.parent.mkdir(parents=True, exist_ok=True)\n                with archive.open(name) as source, target.open("xb") as output:\n                    shutil.copyfileobj(source, output, length=1024 * 1024)\n                if sha256_file(target) != member["sha256"]:\n                    raise ValueError("Extracted source checksum mismatch.")\n\n\ndef load_bundle(path, info, config, contract):\n    path = Path(path)\n    folder = path.is_dir()\n    digest = None\n    if not folder:\n        if not path.is_file():\n            raise FileNotFoundError("Upload the included ZIP or extracted dataset folder first.")\n        if path.stat().st_size > MAX_COMPRESSED or path.stat().st_size != info["zip_bytes"]:\n            raise ValueError("Dataset ZIP size does not match the included release.")\n        digest = sha256_file(path)\n        if not re.fullmatch(r"[0-9a-f]{64}", info["zip_sha256"]) or digest != info["zip_sha256"]:\n            raise ValueError("Dataset ZIP checksum mismatch. Use the data ZIP shipped with these notebooks.")\n    schemas = source_schemas(contract)\n    tables, all_members = {}, []\n    with (_DirectoryArchive(path) if folder else zipfile.ZipFile(path)) as archive:\n        entries = archive.infolist()\n        if len(entries) > MAX_FILES or len({e.filename for e in entries}) != len(entries):\n            raise ValueError("Too many archive entries or duplicate member names.")\n        for entry in entries:\n            safe_member_name(entry.filename)\n            if entry.is_dir() or entry.flag_bits & 1 or stat.S_ISLNK(entry.external_attr >> 16):\n                raise ValueError("Unexpected directory, encrypted entry or symbolic link.")\n            if entry.file_size > MAX_MEMBER:\n                raise ValueError("Dataset member exceeds the uncompressed-size limit.")\n        if sum(e.file_size for e in entries) > MAX_UNCOMPRESSED:\n            raise ValueError("Dataset exceeds the total uncompressed-size limit.")\n        if "manifest.json" not in archive.namelist() or archive.getinfo("manifest.json").file_size > 1024 * 1024:\n            raise ValueError("Dataset manifest is absent or oversized.")\n        manifest_bytes = archive.read("manifest.json")\n        manifest_digest = hashlib.sha256(manifest_bytes).hexdigest()\n        if folder and (not re.fullmatch(r"[0-9a-f]{64}", info.get("manifest_sha256", "")) or\n                       manifest_digest != info["manifest_sha256"]):\n            raise ValueError("Dataset manifest checksum mismatch. Use the extracted data shipped with this notebook.")\n        manifest = json.loads(manifest_bytes)\n        if manifest.get("purpose") != PURPOSE or manifest.get("version") != 1 or manifest.get("synthetic") is not True:\n            raise ValueError("Wrong dataset type.")\n        if manifest["generation_settings"] != generation_settings(config):\n            raise ValueError("Included data does not match the configured synthetic profile/seed.")\n        if manifest["contract_sha256"] != hashlib.sha256(dumps(contract).encode()).hexdigest():\n            raise ValueError("Source contract does not match the bundled data.")\n        listed = {item["path"] for item in manifest["members"]}\n        if len(listed) != len(manifest["members"]) or set(archive.namelist()) != listed | {"manifest.json"}:\n            raise ValueError("Missing, duplicated or unlisted dataset members.")\n        summary = None\n        for item in manifest["members"]:\n            name = safe_member_name(item["path"])\n            entry = archive.getinfo(name)\n            if entry.file_size != item["bytes"]:\n                raise ValueError("Dataset member size mismatch.")\n            raw = archive.read(name)\n            if hashlib.sha256(raw).hexdigest() != item["sha256"]:\n                raise ValueError("Dataset member checksum mismatch.")\n            if name == "summary.json":\n                summary = json.loads(raw)\n                continue\n            if not name.endswith(".parquet"):\n                raise ValueError("Unexpected non-Parquet dataset member.")\n            if name.startswith("source_parquet/"):\n                parts = name.split("/")\n                if len(parts) != 4 or not re.fullmatch(r"part-\\d{2}\\.parquet", parts[-1]):\n                    raise ValueError("Unexpected source path.")\n                table = parts[1] + "." + parts[2]\n                if table not in schemas or item["table"] != table:\n                    raise ValueError("Unexpected source table.")\n            elif name in ("features/historical.parquet", "features/current.parquet"):\n                table = name\n            else:\n                raise ValueError("Unexpected feature path.")\n            arrow = pq.read_table(io.BytesIO(raw))\n            arrow.validate(full=True)\n            if arrow.num_rows != item["rows"] or str(arrow.schema.remove_metadata()) != item["schema"]:\n                raise ValueError("Parquet schema or row-count mismatch.")\n            if table in schemas and not arrow.schema.equals(schemas[table], check_metadata=False):\n                raise ValueError("Parquet does not match its declared Canvas projection.")\n            tables.setdefault(table, []).append(arrow)\n            all_members.append(name)\n    if set(tables) != set(schemas) | {"features/historical.parquet", "features/current.parquet"} or summary is None:\n        raise ValueError("Incomplete source/feature/summary bundle.")\n    if summary.get("synthetic") is not True or summary.get("profile") != "small":\n        raise ValueError("Bundle is not the included small synthetic population.")\n    references = {}\n    for table, definition in contract["tables"].items():\n        data = pa.concat_tables(tables[definition["namespace"] + "." + table]).to_pandas()\n        for column, spec in definition["columns"].items():\n            if spec["type"] == "array<string>":\n                data[column] = data[column].map(lambda value: list(value) if value is not None else None)\n        references[table] = frame(table, data.to_dict("records"), contract)\n    for table, data in references.items():\n        validate_table(table, data, contract, references)\n        if len(data) != summary["observed_source_counts"][table]:\n            raise ValueError("Source count differs from the verified generation summary.")\n    historical = normalize_timestamp_units(tables["features/historical.parquet"][0].to_pandas())\n    current = normalize_timestamp_units(tables["features/current.parquet"][0].to_pandas())\n    validate_feature_frames(historical, current)\n    enrollment_ids = set(references["enrollments"].id)\n    if not (set(historical.enrollment_id) | set(current.enrollment_id)) <= enrollment_ids:\n        raise ValueError("Feature enrollment lacks a source record.")\n    current = current.copy()\n    current["label"] = None\n    current["label_status"] = "current_term_no_truth_published"\n    current["label_available_at"] = pd.NaT\n    features = pd.concat([historical, current], ignore_index=True)\n    features["label_available_at"] = pd.to_datetime(features.label_available_at, utc=True)\n    if len(features) != summary["feature_snapshots"]:\n        raise ValueError("Prepared feature count mismatch.")\n    manifest["verified_input_kind"] = "folder" if folder else "uploaded"\n    manifest["verified_manifest_sha256"] = manifest_digest\n    if digest is not None:\n        manifest["verified_zip_sha256"] = digest\n    summary["input_mode"] = "folder_verified_synthetic_bundle" if folder else "uploaded_verified_synthetic_bundle"\n    summary["prepared_features"] = "Computed from the included synthetic history during dataset packaging, not regenerated on upload."\n    return DatasetBundle(path, manifest, features, summary,\n                         {name: pa.concat_tables(parts) for name, parts in tables.items()})\n', 'aews.runtime': '"""Read-only Fabric runtime and publication checks. No cloud session is created."""\nimport importlib\nfrom importlib.metadata import PackageNotFoundError, version\nfrom pathlib import Path\nimport sys\n\nPURPOSE = "academic_early_warning_current_publication"\nCURRENT_MANIFEST = "Files/model_artifacts/current.json"\nSCHEMAS = ("canvas", "canvas_logs", "sim_history", "sim_support", "silver", "gold")\nTABLES = ("silver.historical_weekly_features", "silver.current_weekly_features",\n          "gold.student_course_predictions", "gold.prediction_explanations",\n          "gold.model_evaluation", "gold.data_quality_summary")\n\n\ndef qualified(config, schema, table=None):\n    if schema not in SCHEMAS and schema != "dbo":\n        raise ValueError("Unexpected schema.")\n    pieces = [config["intended_lakehouse"], schema] + ([table] if table else [])\n    if any(not p or any(c in p for c in "`\\r\\n/\\\\") for p in pieces):\n        raise ValueError("Unsafe catalog name.")\n    return ".".join("`" + piece + "`" for piece in pieces)\n\n\ndef _mlflow_version():\n    try:\n        return version("mlflow")\n    except PackageNotFoundError:\n        module = importlib.import_module("mlflow")\n        value = getattr(module, "__version__", None)\n        if not isinstance(value, str) or not value:\n            raise RuntimeError("MLflow is required for modeling, but its version is unavailable.")\n        return value\n\n\ndef _delta_version(spark):\n    importlib.import_module("delta.tables")\n    spark._jvm.java.lang.Thread.currentThread().getContextClassLoader().loadClass("io.delta.tables.DeltaTable")\n    try:\n        value = version("delta-spark")\n    except PackageNotFoundError:\n        print("Fabric-provided Delta Python and Spark classes are available; exact Delta version metadata is unavailable.")\n        return None\n    if not value.startswith("3.2."):\n        raise RuntimeError(f"Expected Delta 3.2, observed {value}.")\n    return value\n\n\ndef validate_session(spark, config, *, require_model=True):\n    if spark is None:\n        raise RuntimeError("Attach a Lakehouse and use the existing Fabric Spark session.")\n    if sys.version_info[:2] != (3, 11):\n        raise RuntimeError("This starter targets Python 3.11 in Fabric Runtime 1.3.")\n    if not spark.version.startswith("3.5."):\n        raise RuntimeError("This starter targets Spark 3.5 in Fabric Runtime 1.3.")\n    import notebookutils\n    from pyspark.sql import SparkSession\n    if SparkSession.getActiveSession() is None:\n        raise RuntimeError("No active Spark session; this starter never creates one.")\n    context = dict(notebookutils.runtime.context)\n    if not context.get("defaultLakehouseName") or not context.get("defaultLakehouseWorkspaceName"):\n        raise RuntimeError(\n            "The running session has no default Lakehouse. Set the intended Lakehouse as default, "\n            "stop/restart the notebook session, then rerun all cells.")\n    for key, field in (("currentWorkspaceName", "intended_workspace"),\n                       ("defaultLakehouseName", "intended_lakehouse"),\n                       ("defaultLakehouseWorkspaceName", "intended_workspace")):\n        if context.get(key) != config[field]:\n            raise RuntimeError(\n                f"{key} is not the configured target: expected {config[field]!r}, observed {context.get(key)!r}. "\n                "Open the notebook in the intended workspace, right-click its Lakehouse -> Set as default lakehouse, "\n                "then stop/restart the notebook session and rerun all cells.")\n    expected = {"numpy": "1.26.4", "pandas": "2.1.4", "scikit-learn": "1.2.2", "pyarrow": "14.0.2"}\n    observed = {name: version(name) for name in expected}\n    if require_model:\n        expected["mlflow"] = "2.12.2"\n        observed["mlflow"] = _mlflow_version()\n    for name, required in expected.items():\n        if observed[name] != required:\n            raise RuntimeError(f"Expected {name} {required}, observed {observed[name]}; validate compatibility before changing the guard.")\n    if require_model:\n        importlib.import_module("mlflow.sklearn")\n    delta_version = _delta_version(spark)\n    # Probe the configured namespace directly; SHOW SCHEMAS name formatting varies by catalog.\n    spark.sql("SHOW TABLES IN " + qualified(config, "dbo")).collect()\n    root = Path("/") / "lakehouse" / "default" / "Files"\n    if not root.is_dir():\n        raise RuntimeError("Default Lakehouse Files mount unavailable.")\n    return root, {**observed, "python": ".".join(map(str, sys.version_info[:3])),\n                  "spark": spark.version, "delta": delta_version,\n                  "delta_capability_verified": True,\n                  "delta_version_source": "package_metadata" if delta_version is not None else "unavailable"}\n\n\ndef validate_manifest(manifest):\n    if manifest.get("purpose") != PURPOSE or manifest.get("publication_state") != "complete":\n        raise RuntimeError("Publication is absent, incomplete or unrelated.")\n    if manifest.get("source_profile") not in ("small", "full") or manifest.get("registered") is not False:\n        raise RuntimeError("Exploration requires a recognized synthetic unregistered publication.")\n    required = ("model_version", "mlflow_run_id", "code_sha256", "source_dataset_version",\n                "train_data_sha256", "publication_archive")\n    if any(not isinstance(manifest.get(key), str) or not manifest[key] for key in required):\n        raise RuntimeError("Publication lacks immutable provenance.")\n    if manifest.get("schemas") != {"silver": "silver", "gold": "gold"}:\n        raise RuntimeError("Publication schemas do not match.")\n    if set(manifest.get("table_paths", {})) != set(TABLES):\n        raise RuntimeError("Publication table snapshots are incomplete.")\n    root = manifest["publication_archive"]\n    if not root.startswith("Files/model_artifacts/") or ".." in root or "\\\\" in root:\n        raise RuntimeError("Unsafe publication archive path.")\n    for table, path in manifest["table_paths"].items():\n        if path != root + "/tables/" + table.replace(".", "/"):\n            raise RuntimeError("Table path is outside its immutable publication archive.")\n    if int(manifest.get("current_predictions", 0)) <= 0:\n        raise RuntimeError("Publication has no predictions.")\n    return manifest\n', 'aews.spark_tables': 'import pandas as pd\n\n\ndef _spark_schema(arrow_schema, nullable=False):\n    import pyarrow as pa\n    from pyspark.sql.types import (ArrayType, BooleanType, DoubleType, IntegerType,\n                                   LongType, StringType, StructField, StructType, TimestampType)\n    fields = []\n    for field in arrow_schema:\n        t = field.type\n        if pa.types.is_int64(t):\n            spark_type = LongType()\n        elif pa.types.is_int32(t):\n            spark_type = IntegerType()\n        elif pa.types.is_boolean(t):\n            spark_type = BooleanType()\n        elif pa.types.is_floating(t):\n            spark_type = DoubleType()\n        elif pa.types.is_timestamp(t):\n            spark_type = TimestampType()\n        elif pa.types.is_list(t):\n            spark_type = ArrayType(StringType(), containsNull=nullable)\n        elif pa.types.is_string(t) or pa.types.is_large_string(t):\n            spark_type = StringType()\n        else:\n            raise TypeError(f"Unsupported explicit Arrow-to-Spark type: {t}")\n        fields.append(StructField(field.name, spark_type, field.nullable or nullable))\n    return StructType(fields)\n\n\n\ndef _pandas_to_spark(spark, data):\n    import pyarrow as pa\n    # Explicit schema preserves all-null columns (e.g. no registry version).\n    fields = []\n    string_names = {"registered_version", "mlflow_run_id", "label", "label_status", "model_status"}\n    for name in data:\n        dtype = data[name].dtype\n        if pd.api.types.is_datetime64_any_dtype(dtype):\n            typ = pa.timestamp("us", tz="UTC")\n        elif pd.api.types.is_bool_dtype(dtype):\n            typ = pa.bool_()\n        elif pd.api.types.is_integer_dtype(dtype):\n            typ = pa.int64()\n        elif pd.api.types.is_float_dtype(dtype):\n            typ = pa.float64()\n        elif name in string_names or pd.api.types.is_string_dtype(dtype) or dtype == object:\n            typ = pa.string()\n        else:\n            raise TypeError(f"Unsupported output type: {name} {dtype}")\n        fields.append(pa.field(name, typ))\n    schema = pa.schema(fields)\n    copied = data.copy()\n    for name in copied:\n        if pd.api.types.is_datetime64_any_dtype(copied[name].dtype):\n            copied[name] = pd.to_datetime(copied[name], utc=True).dt.floor("us")\n    arrow = pa.Table.from_pandas(copied, schema=schema, preserve_index=False, safe=True)\n    return _arrow_to_spark(spark, arrow)\n\n\ndef _arrow_to_spark(spark, arrow, nullable=False):\n    records = []\n    for row in arrow.to_pylist():\n        records.append(tuple(v.to_pydatetime() if isinstance(v, pd.Timestamp) else v for v in row.values()))\n    return spark.createDataFrame(records, schema=_spark_schema(arrow.schema, nullable=nullable))\n', 'aews.storage': 'import json\n\n\ndef canonical_json(value):\n    return json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False)\n\n\ndef read_json(fs, path):\n    return json.loads(fs.head(path, 16 * 1024 * 1024))\n\n\ndef write_once_json(fs, path, value):\n    if fs.exists(path):\n        if read_json(fs, path) != value:\n            raise RuntimeError(f"Unknown or conflicting existing manifest: {path}")\n        return\n    parent = path.rsplit("/", 1)[0]\n    if not fs.mkdirs(parent):\n        raise RuntimeError(f"Could not create manifest parent: {parent}")\n    if not fs.put(path, canonical_json(value), overwrite=False):\n        raise RuntimeError(f"Could not persist manifest: {path}")\n    if read_json(fs, path) != value:\n        raise RuntimeError(f"Manifest read-back mismatch: {path}")\n\n\ndef schema_signature(frame):\n    return canonical_json(frame.schema.jsonValue())\n\n\ndef require_equivalent(left, right, description):\n    if schema_signature(left) != schema_signature(right):\n        raise RuntimeError(f"Schema mismatch: {description}")\n    left_count, right_count = left.count(), right.count()\n    if left_count != right_count:\n        raise RuntimeError(f"Row count mismatch: {description}: {left_count} != {right_count}")\n    if left.exceptAll(right).limit(1).count() or right.exceptAll(left).limit(1).count():\n        raise RuntimeError(f"Row content mismatch: {description}")\n    return left_count\n\n\ndef archive_delta(spark, fs, frame, path):\n    if not fs.exists(path):\n        frame.write.format("delta").mode("errorifexists").save(path)\n    archived = spark.read.format("delta").load(path)\n    require_equivalent(frame, archived, path)\n    return archived\n\n\n', 'aews.table_import': '"""Create the named input tables from a verified folder, without training or replacement."""\nimport json\n\nfrom aews.data_bundle import load_lakehouse_input, source_schemas\nfrom aews.runtime import qualified, validate_session\nfrom aews.spark_tables import _arrow_to_spark\nfrom aews.storage import require_equivalent\n\n\nFEATURE_TARGETS = {\n    "features/current.parquet": "silver.current_weekly_features",\n    "features/historical.parquet": "silver.historical_weekly_features",\n}\n\n\ndef table_plan(bundle, contract):\n    expected = set(source_schemas(contract)) | set(FEATURE_TARGETS)\n    if set(bundle.tables) != expected:\n        raise ValueError("The verified dataset is missing expected input tables.")\n    return {FEATURE_TARGETS.get(name, name): table for name, table in sorted(bundle.tables.items())}\n\n\ndef _delta_frame(spark, arrow):\n    # Delta readers can expose nullable columns even for non-null source fields.\n    # The source contract has already checked required values before any write.\n    return _arrow_to_spark(spark, arrow, nullable=True)\n\n\ndef create_input_tables(spark, config, plan):\n    results, existing = [], set()\n    for name, arrow in plan.items():\n        schema, table = name.split(".")\n        target = qualified(config, schema, table)\n        if spark.catalog.tableExists(target):\n            print(f"Checking existing table {name}; mismatches stop the import without replacement.")\n            count = require_equivalent(_delta_frame(spark, arrow), spark.table(target), target)\n            results.append({"table": name, "status": "already_matches", "rows": count})\n            existing.add(name)\n    for schema in sorted({name.split(".")[0] for name in plan}):\n        spark.sql("CREATE SCHEMA IF NOT EXISTS " + qualified(config, schema))\n    for name, arrow in plan.items():\n        if name in existing:\n            continue\n        schema, table = name.split(".")\n        target = qualified(config, schema, table)\n        print(f"Creating {name} from the verified input records.")\n        frame = _delta_frame(spark, arrow)\n        frame.write.format("delta").mode("errorifexists").saveAsTable(target)\n        count = require_equivalent(frame, spark.table(target), target)\n        results.append({"table": name, "status": "created", "rows": count})\n        print(json.dumps(results[-1]))\n    results.sort(key=lambda row: row["table"])\n    print(json.dumps({"status": "tables_loaded", "tables": len(results), "training_performed": False}))\n    return results\n\n\ndef load_folder_tables(spark, source_config, contract, source_folder, bundle_info):\n    root, _ = validate_session(spark, source_config, require_model=False)\n    bundle = load_lakehouse_input(root, "folder", source_folder, bundle_info, source_config, contract)\n    plan = table_plan(bundle, contract)\n    print(json.dumps({"status": "folder_verified", "input_tables": len(plan), "writes_performed": False}))\n    return create_input_tables(spark, source_config, plan)\n'}
EMBEDDED_PACKAGES = ['aews', 'aews.core']
for name, source in EMBEDDED_MODULES.items():
    module = types.ModuleType(name)
    module.__file__ = '<embedded:' + name + '>'
    module.__package__ = name if name in EMBEDDED_PACKAGES else name.rpartition('.')[0]
    if name in EMBEDDED_PACKAGES:
        module.__path__ = []
    sys.modules[name] = module
    if '.' in name:
        parent, _, child = name.rpartition('.')
        setattr(sys.modules[parent], child, module)
    exec(compile(source, module.__file__, 'exec'), module.__dict__)
print('Self-contained Academic Early Warning System source loaded.')


In [ ]:
import json
SOURCE_CONFIG = json.loads('{"seed": 271828, "mode": "small", "profiles": {"small": {"students": 400, "courses_per_term": 8, "enrollments_per_course": 75, "assignments_per_course": 20, "sections_per_course": 3}, "full": {"students": 5000, "courses_per_term": 80, "enrollments_per_course": 150, "assignments_per_course": 20, "sections_per_course": 3}}, "terms": 5, "term_days": 84, "cutoff_weeks": [4], "current_week": 4, "intended_workspace": "Academic Early Warning System", "intended_lakehouse": "academic_early_warning", "experiment_name": "Academic Early Warning System Training", "notice": "Synthetic course outcomes only. No real Canvas ingestion or accepted model."}')
SOURCE_CONTRACT = json.loads('{"version": "canvas-selected-projection-1", "public_sources": {"canvas": {"url": "https://developerdocs.instructure.com/services/dap/dataset/dataset-namespaces/dataset-canvas.md", "sha256": "f76419a3bad3b0515c584953c47cc6b4274ad3fa96ea031df2719fea20dc2d46"}, "types": {"url": "https://developerdocs.instructure.com/services/dap/dataset/dataset-namespaces/dataset-canvas/dataset-canvas-types.md", "sha256": "bf91631fc1001cc3a13c61ee001598da938d4cfc9bd4fe89c07433407ce806a2"}, "canvas_logs": {"url": "https://developerdocs.instructure.com/services/dap/dataset/dataset-namespaces/dataset-canvaslogs.md", "sha256": "ebe0106e785048bfe998fe1aaf48c2dba64e431021cd38ef771efbf5d5256f7a"}}, "scope": "Selected projection only, not full Canvas replication. All omitted fields and all other tables are not simulated.", "tables": {"accounts": {"namespace": "canvas", "columns": {"id": {"published_type": "int64", "type": "int64", "nullable": false, "primary_key": true, "reference": null, "enum": null, "max_length": null, "format": null}, "name": {"published_type": "Annotated[str, MaxLength(255)] | None", "type": "string", "nullable": true, "primary_key": false, "reference": null, "enum": null, "max_length": 255, "format": null}, "created_at": {"published_type": "datetime", "type": "datetime", "nullable": false, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}, "updated_at": {"published_type": "datetime", "type": "datetime", "nullable": false, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}, "workflow_state": {"published_type": "[accounts__workflow_state](https://developerdocs.instructure.com/services/dap/dataset/dataset-namespaces/pages/HxvoImWoy0bxSacLiOmu#dap_schemas.canvas.accounts__workflow_state)", "type": "string", "nullable": false, "primary_key": false, "reference": null, "enum": ["__dap_unspecified__", "active", "deleted", "suspended"], "max_length": null, "format": null}}}, "users": {"namespace": "canvas", "columns": {"id": {"published_type": "int64", "type": "int64", "nullable": false, "primary_key": true, "reference": null, "enum": null, "max_length": null, "format": null}, "name": {"published_type": "Annotated[str, MaxLength(255)] | None", "type": "string", "nullable": true, "primary_key": false, "reference": null, "enum": null, "max_length": 255, "format": null}, "created_at": {"published_type": "datetime", "type": "datetime", "nullable": false, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}, "updated_at": {"published_type": "datetime", "type": "datetime", "nullable": false, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}, "workflow_state": {"published_type": "[users__workflow_state](https://developerdocs.instructure.com/services/dap/dataset/dataset-namespaces/pages/HxvoImWoy0bxSacLiOmu#dap_schemas.canvas.users__workflow_state)", "type": "string", "nullable": false, "primary_key": false, "reference": null, "enum": ["__dap_unspecified__", "creation_pending", "deleted", "pending_approval", "pre_registered", "registered"], "max_length": null, "format": null}}}, "enrollment_terms": {"namespace": "canvas", "columns": {"id": {"published_type": "int64", "type": "int64", "nullable": false, "primary_key": true, "reference": null, "enum": null, "max_length": null, "format": null}, "name": {"published_type": "Annotated[str, MaxLength(255)] | None", "type": "string", "nullable": true, "primary_key": false, "reference": null, "enum": null, "max_length": 255, "format": null}, "start_at": {"published_type": "datetime | None", "type": "datetime", "nullable": true, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}, "end_at": {"published_type": "datetime | None", "type": "datetime", "nullable": true, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}, "created_at": {"published_type": "datetime", "type": "datetime", "nullable": false, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}, "updated_at": {"published_type": "datetime", "type": "datetime", "nullable": false, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}, "workflow_state": {"published_type": "[enrollment_terms__workflow_state](https://developerdocs.instructure.com/services/dap/dataset/dataset-namespaces/pages/HxvoImWoy0bxSacLiOmu#dap_schemas.canvas.enrollment_terms__workflow_state)", "type": "string", "nullable": false, "primary_key": false, "reference": null, "enum": ["__dap_unspecified__", "active", "deleted"], "max_length": null, "format": null}}}, "courses": {"namespace": "canvas", "columns": {"id": {"published_type": "int64", "type": "int64", "nullable": false, "primary_key": true, "reference": null, "enum": null, "max_length": null, "format": null}, "name": {"published_type": "Annotated[str, MaxLength(255)] | None", "type": "string", "nullable": true, "primary_key": false, "reference": null, "enum": null, "max_length": 255, "format": null}, "account_id": {"published_type": "[accounts](#dap_schemas.canvas.accounts)", "type": "int64", "nullable": false, "primary_key": false, "reference": "accounts", "enum": null, "max_length": null, "format": null}, "enrollment_term_id": {"published_type": "[enrollment_terms](#dap_schemas.canvas.enrollment_terms)", "type": "int64", "nullable": false, "primary_key": false, "reference": "enrollment_terms", "enum": null, "max_length": null, "format": null}, "start_at": {"published_type": "datetime | None", "type": "datetime", "nullable": true, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}, "conclude_at": {"published_type": "datetime | None", "type": "datetime", "nullable": true, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}, "created_at": {"published_type": "datetime", "type": "datetime", "nullable": false, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}, "updated_at": {"published_type": "datetime", "type": "datetime", "nullable": false, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}, "workflow_state": {"published_type": "[courses__workflow_state](https://developerdocs.instructure.com/services/dap/dataset/dataset-namespaces/pages/HxvoImWoy0bxSacLiOmu#dap_schemas.canvas.courses__workflow_state)", "type": "string", "nullable": false, "primary_key": false, "reference": null, "enum": ["__dap_unspecified__", "created", "claimed", "available", "completed", "deleted"], "max_length": null, "format": null}}}, "course_sections": {"namespace": "canvas", "columns": {"id": {"published_type": "int64", "type": "int64", "nullable": false, "primary_key": true, "reference": null, "enum": null, "max_length": null, "format": null}, "name": {"published_type": "Annotated[str, MaxLength(255)]", "type": "string", "nullable": false, "primary_key": false, "reference": null, "enum": null, "max_length": 255, "format": null}, "course_id": {"published_type": "[courses](#dap_schemas.canvas.courses)", "type": "int64", "nullable": false, "primary_key": false, "reference": "courses", "enum": null, "max_length": null, "format": null}, "enrollment_term_id": {"published_type": "[enrollment_terms](#dap_schemas.canvas.enrollment_terms) | None", "type": "int64", "nullable": true, "primary_key": false, "reference": "enrollment_terms", "enum": null, "max_length": null, "format": null}, "start_at": {"published_type": "datetime | None", "type": "datetime", "nullable": true, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}, "end_at": {"published_type": "datetime | None", "type": "datetime", "nullable": true, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}, "created_at": {"published_type": "datetime", "type": "datetime", "nullable": false, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}, "updated_at": {"published_type": "datetime", "type": "datetime", "nullable": false, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}, "workflow_state": {"published_type": "[course_sections__workflow_state](https://developerdocs.instructure.com/services/dap/dataset/dataset-namespaces/pages/HxvoImWoy0bxSacLiOmu#dap_schemas.canvas.course_sections__workflow_state)", "type": "string", "nullable": false, "primary_key": false, "reference": null, "enum": ["__dap_unspecified__", "active", "deleted"], "max_length": null, "format": null}}}, "enrollments": {"namespace": "canvas", "columns": {"id": {"published_type": "int64", "type": "int64", "nullable": false, "primary_key": true, "reference": null, "enum": null, "max_length": null, "format": null}, "user_id": {"published_type": "[users](#dap_schemas.canvas.users)", "type": "int64", "nullable": false, "primary_key": false, "reference": "users", "enum": null, "max_length": null, "format": null}, "course_id": {"published_type": "[courses](#dap_schemas.canvas.courses)", "type": "int64", "nullable": false, "primary_key": false, "reference": "courses", "enum": null, "max_length": null, "format": null}, "course_section_id": {"published_type": "[course_sections](#dap_schemas.canvas.course_sections)", "type": "int64", "nullable": false, "primary_key": false, "reference": "course_sections", "enum": null, "max_length": null, "format": null}, "type": {"published_type": "[enrollments__type](https://developerdocs.instructure.com/services/dap/dataset/dataset-namespaces/pages/HxvoImWoy0bxSacLiOmu#dap_schemas.canvas.enrollments__type)", "type": "string", "nullable": false, "primary_key": false, "reference": null, "enum": ["__dap_unspecified__", "StudentEnrollment", "TeacherEnrollment", "TaEnrollment", "DesignerEnrollment", "ObserverEnrollment", "StudentViewEnrollment"], "max_length": null, "format": null}, "start_at": {"published_type": "datetime | None", "type": "datetime", "nullable": true, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}, "end_at": {"published_type": "datetime | None", "type": "datetime", "nullable": true, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}, "created_at": {"published_type": "datetime", "type": "datetime", "nullable": false, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}, "updated_at": {"published_type": "datetime", "type": "datetime", "nullable": false, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}, "workflow_state": {"published_type": "[enrollments__workflow_state](https://developerdocs.instructure.com/services/dap/dataset/dataset-namespaces/pages/HxvoImWoy0bxSacLiOmu#dap_schemas.canvas.enrollments__workflow_state)", "type": "string", "nullable": false, "primary_key": false, "reference": null, "enum": ["__dap_unspecified__", "creation_pending", "invited", "rejected", "active", "inactive", "completed", "deleted"], "max_length": null, "format": null}}}, "assignment_groups": {"namespace": "canvas", "columns": {"id": {"published_type": "int64", "type": "int64", "nullable": false, "primary_key": true, "reference": null, "enum": null, "max_length": null, "format": null}, "name": {"published_type": "Annotated[str, MaxLength(255)] | None", "type": "string", "nullable": true, "primary_key": false, "reference": null, "enum": null, "max_length": 255, "format": null}, "context_id": {"published_type": "[courses](#dap_schemas.canvas.courses)", "type": "int64", "nullable": false, "primary_key": false, "reference": "courses", "enum": null, "max_length": null, "format": null}, "context_type": {"published_type": "[assignment_groups__context_type](https://developerdocs.instructure.com/services/dap/dataset/dataset-namespaces/pages/HxvoImWoy0bxSacLiOmu#dap_schemas.canvas.assignment_groups__context_type)", "type": "string", "nullable": false, "primary_key": false, "reference": null, "enum": ["__dap_unspecified__", "Course"], "max_length": null, "format": null}, "group_weight": {"published_type": "float64 | None", "type": "float64", "nullable": true, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}, "created_at": {"published_type": "datetime", "type": "datetime", "nullable": false, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}, "updated_at": {"published_type": "datetime", "type": "datetime", "nullable": false, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}, "workflow_state": {"published_type": "[assignment_groups__workflow_state](https://developerdocs.instructure.com/services/dap/dataset/dataset-namespaces/pages/HxvoImWoy0bxSacLiOmu#dap_schemas.canvas.assignment_groups__workflow_state)", "type": "string", "nullable": false, "primary_key": false, "reference": null, "enum": ["__dap_unspecified__", "available", "deleted", "active"], "max_length": null, "format": null}}}, "assignments": {"namespace": "canvas", "columns": {"id": {"published_type": "int64", "type": "int64", "nullable": false, "primary_key": true, "reference": null, "enum": null, "max_length": null, "format": null}, "title": {"published_type": "Annotated[str, MaxLength(255)] | None", "type": "string", "nullable": true, "primary_key": false, "reference": null, "enum": null, "max_length": 255, "format": null}, "context_id": {"published_type": "[courses](#dap_schemas.canvas.courses)", "type": "int64", "nullable": false, "primary_key": false, "reference": "courses", "enum": null, "max_length": null, "format": null}, "context_type": {"published_type": "[assignments__context_type](https://developerdocs.instructure.com/services/dap/dataset/dataset-namespaces/pages/HxvoImWoy0bxSacLiOmu#dap_schemas.canvas.assignments__context_type)", "type": "string", "nullable": false, "primary_key": false, "reference": null, "enum": ["__dap_unspecified__", "Course"], "max_length": null, "format": null}, "assignment_group_id": {"published_type": "[assignment_groups](#dap_schemas.canvas.assignment_groups) | None", "type": "int64", "nullable": true, "primary_key": false, "reference": "assignment_groups", "enum": null, "max_length": null, "format": null}, "due_at": {"published_type": "datetime | None", "type": "datetime", "nullable": true, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}, "unlock_at": {"published_type": "datetime | None", "type": "datetime", "nullable": true, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}, "points_possible": {"published_type": "float64 | None", "type": "float64", "nullable": true, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}, "submission_types": {"published_type": "Annotated[list[[assignments__submission_types](https://developerdocs.instructure.com/services/dap/dataset/dataset-namespaces/pages/HxvoImWoy0bxSacLiOmu#dap_schemas.canvas.assignments__submission_types)], SpecialConversion()] | None", "type": "array<string>", "nullable": true, "primary_key": false, "reference": null, "enum": ["__dap_unspecified__", "discussion_topic", "online_quiz", "on_paper", "none", "external_tool", "online", "online_text_entry", "online_url", "online_upload", "media_recording", "not_graded", "wiki_page", "student_annotation", "default_external_tool"], "max_length": null, "format": null}, "grading_type": {"published_type": "[assignments__grading_type](https://developerdocs.instructure.com/services/dap/dataset/dataset-namespaces/pages/HxvoImWoy0bxSacLiOmu#dap_schemas.canvas.assignments__grading_type) | None", "type": "string", "nullable": true, "primary_key": false, "reference": null, "enum": ["__dap_unspecified__", "gpa_scale", "letter_grade", "not_graded", "pass_fail", "percent", "points", "default"], "max_length": null, "format": null}, "omit_from_final_grade": {"published_type": "bool", "type": "bool", "nullable": false, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}, "created_at": {"published_type": "datetime | None", "type": "datetime", "nullable": true, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}, "updated_at": {"published_type": "datetime | None", "type": "datetime", "nullable": true, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}, "workflow_state": {"published_type": "[assignments__workflow_state](https://developerdocs.instructure.com/services/dap/dataset/dataset-namespaces/pages/HxvoImWoy0bxSacLiOmu#dap_schemas.canvas.assignments__workflow_state)", "type": "string", "nullable": false, "primary_key": false, "reference": null, "enum": ["__dap_unspecified__", "unpublished", "published", "deleted", "duplicating", "fail_to_import", "failed_to_duplicate", "failed_to_import", "failed_to_migrate", "importing", "migrating", "outcome_alignment_cloning", "failed_to_clone_outcome_alignment"], "max_length": null, "format": null}}}, "assignment_overrides": {"namespace": "canvas", "columns": {"id": {"published_type": "int64", "type": "int64", "nullable": false, "primary_key": true, "reference": null, "enum": null, "max_length": null, "format": null}, "assignment_id": {"published_type": "[assignments](#dap_schemas.canvas.assignments) | None", "type": "int64", "nullable": true, "primary_key": false, "reference": "assignments", "enum": null, "max_length": null, "format": null}, "set_type": {"published_type": "[assignment_overrides__set_type](https://developerdocs.instructure.com/services/dap/dataset/dataset-namespaces/pages/HxvoImWoy0bxSacLiOmu#dap_schemas.canvas.assignment_overrides__set_type) | None", "type": "string", "nullable": true, "primary_key": false, "reference": null, "enum": ["__dap_unspecified__", "Group", "CourseSection", "ADHOC", "Noop", "Course"], "max_length": null, "format": null}, "set_id": {"published_type": "None | [groups](#dap_schemas.canvas.groups) | [course_sections](#dap_schemas.canvas.course_sections)", "type": "int64", "nullable": true, "primary_key": true, "reference": "course_sections", "enum": null, "max_length": null, "format": null}, "due_at": {"published_type": "datetime | None", "type": "datetime", "nullable": true, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}, "due_at_overridden": {"published_type": "bool", "type": "bool", "nullable": false, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}, "title": {"published_type": "Annotated[str, MaxLength(255)]", "type": "string", "nullable": false, "primary_key": false, "reference": null, "enum": null, "max_length": 255, "format": null}, "created_at": {"published_type": "datetime", "type": "datetime", "nullable": false, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}, "updated_at": {"published_type": "datetime", "type": "datetime", "nullable": false, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}, "workflow_state": {"published_type": "[assignment_overrides__workflow_state](https://developerdocs.instructure.com/services/dap/dataset/dataset-namespaces/pages/HxvoImWoy0bxSacLiOmu#dap_schemas.canvas.assignment_overrides__workflow_state)", "type": "string", "nullable": false, "primary_key": false, "reference": null, "enum": ["__dap_unspecified__", "active", "deleted"], "max_length": null, "format": null}}}, "assignment_override_students": {"namespace": "canvas", "columns": {"id": {"published_type": "int64", "type": "int64", "nullable": false, "primary_key": true, "reference": null, "enum": null, "max_length": null, "format": null}, "user_id": {"published_type": "[users](#dap_schemas.canvas.users)", "type": "int64", "nullable": false, "primary_key": false, "reference": "users", "enum": null, "max_length": null, "format": null}, "assignment_id": {"published_type": "[assignments](#dap_schemas.canvas.assignments) | None", "type": "int64", "nullable": true, "primary_key": false, "reference": "assignments", "enum": null, "max_length": null, "format": null}, "assignment_override_id": {"published_type": "[assignment_overrides](#dap_schemas.canvas.assignment_overrides)", "type": "int64", "nullable": false, "primary_key": false, "reference": "assignment_overrides", "enum": null, "max_length": null, "format": null}, "created_at": {"published_type": "datetime", "type": "datetime", "nullable": false, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}, "updated_at": {"published_type": "datetime", "type": "datetime", "nullable": false, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}, "workflow_state": {"published_type": "[assignment_override_students__workflow_state](https://developerdocs.instructure.com/services/dap/dataset/dataset-namespaces/pages/HxvoImWoy0bxSacLiOmu#dap_schemas.canvas.assignment_override_students__workflow_state)", "type": "string", "nullable": false, "primary_key": false, "reference": null, "enum": ["__dap_unspecified__", "active", "deleted"], "max_length": null, "format": null}}}, "submissions": {"namespace": "canvas", "columns": {"id": {"published_type": "int64", "type": "int64", "nullable": false, "primary_key": true, "reference": null, "enum": null, "max_length": null, "format": null}, "user_id": {"published_type": "[users](#dap_schemas.canvas.users)", "type": "int64", "nullable": false, "primary_key": false, "reference": "users", "enum": null, "max_length": null, "format": null}, "assignment_id": {"published_type": "[assignments](#dap_schemas.canvas.assignments)", "type": "int64", "nullable": false, "primary_key": false, "reference": "assignments", "enum": null, "max_length": null, "format": null}, "submitted_at": {"published_type": "datetime | None", "type": "datetime", "nullable": true, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}, "graded_at": {"published_type": "datetime | None", "type": "datetime", "nullable": true, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}, "score": {"published_type": "float64 | None", "type": "float64", "nullable": true, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}, "attempt": {"published_type": "int32 | None", "type": "int32", "nullable": true, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}, "excused": {"published_type": "bool | None", "type": "bool", "nullable": true, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}, "submission_type": {"published_type": "[submissions__submission_type](https://developerdocs.instructure.com/services/dap/dataset/dataset-namespaces/pages/HxvoImWoy0bxSacLiOmu#dap_schemas.canvas.submissions__submission_type) | None", "type": "string", "nullable": true, "primary_key": false, "reference": null, "enum": ["__dap_unspecified__", "basic_lti_launch", "discussion_topic", "external_tool", "media_recording", "online_file_upload", "online_quiz", "online_text_entry", "online_upload", "online_url", "student_annotation"], "max_length": null, "format": null}, "created_at": {"published_type": "datetime | None", "type": "datetime", "nullable": true, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}, "updated_at": {"published_type": "datetime | None", "type": "datetime", "nullable": true, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}, "workflow_state": {"published_type": "[submissions__workflow_state](https://developerdocs.instructure.com/services/dap/dataset/dataset-namespaces/pages/HxvoImWoy0bxSacLiOmu#dap_schemas.canvas.submissions__workflow_state)", "type": "string", "nullable": false, "primary_key": false, "reference": null, "enum": ["__dap_unspecified__", "graded", "pending_review", "submitted", "unsubmitted", "deleted"], "max_length": null, "format": null}}}, "scores": {"namespace": "canvas", "columns": {"id": {"published_type": "int64", "type": "int64", "nullable": false, "primary_key": true, "reference": null, "enum": null, "max_length": null, "format": null}, "enrollment_id": {"published_type": "[enrollments](#dap_schemas.canvas.enrollments)", "type": "int64", "nullable": false, "primary_key": false, "reference": "enrollments", "enum": null, "max_length": null, "format": null}, "course_score": {"published_type": "bool", "type": "bool", "nullable": false, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}, "current_score": {"published_type": "float64 | None", "type": "float64", "nullable": true, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}, "final_score": {"published_type": "float64 | None", "type": "float64", "nullable": true, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}, "updated_at": {"published_type": "datetime | None", "type": "datetime", "nullable": true, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}}}, "web_logs": {"namespace": "canvas_logs", "columns": {"id": {"published_type": "UUID", "type": "string", "nullable": false, "primary_key": true, "reference": null, "enum": null, "max_length": null, "format": "uuid"}, "timestamp": {"published_type": "datetime", "type": "datetime", "nullable": false, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}, "user_id": {"published_type": "[canvas.users](https://developerdocs.instructure.com/services/dap/dataset/dataset-namespaces/pages/zdvm4Hvd7IRbvpXEbjzE#dap_schemas.canvas.users) | None", "type": "int64", "nullable": true, "primary_key": false, "reference": "users", "enum": null, "max_length": null, "format": null}, "course_id": {"published_type": "[canvas.courses](https://developerdocs.instructure.com/services/dap/dataset/dataset-namespaces/pages/zdvm4Hvd7IRbvpXEbjzE#dap_schemas.canvas.courses) | None", "type": "int64", "nullable": true, "primary_key": false, "reference": "courses", "enum": null, "max_length": null, "format": null}, "url": {"published_type": "str", "type": "string", "nullable": false, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}, "http_method": {"published_type": "[HTTPMethod](#dap_schemas.canvas_logs.httpmethod)", "type": "string", "nullable": false, "primary_key": false, "reference": null, "enum": ["CONNECT", "DELETE", "GET", "HEAD", "OPTIONS", "PATCH", "POST", "PUT", "TRACE", "SEARCH", "PROPFIND", "LOCK", "REPORT", "PURGE", "MKCOL", "PROPPATCH", "CHECKOUT", "UNLOCK", "MERGE", "NOTIFY", "MOVE", "COPY", "UNSUBSCRIBE", "MKACTIVITY", "SUBSCRIBE", "ACL"], "max_length": null, "format": null}, "http_status": {"published_type": "[HTTPStatus](#dap_schemas.canvas_logs.httpstatus)", "type": "int32", "nullable": false, "primary_key": false, "reference": null, "enum": [100, 101, 102, 103, 200, 201, 202, 203, 204, 205, 206, 207, 208, 226, 300, 301, 302, 303, 304, 305, 307, 308, 400, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 421, 422, 423, 424, 425, 426, 428, 429, 431, 451, 500, 501, 502, 503, 504, 505, 506, 507, 508, 510, 511], "max_length": null, "format": null}, "http_version": {"published_type": "[HTTPVersion](#dap_schemas.canvas_logs.httpversion)", "type": "string", "nullable": false, "primary_key": false, "reference": null, "enum": ["0.9", "1.0", "1.1", "2.0"], "max_length": null, "format": null}, "remote_ip": {"published_type": "IPv4Address | IPv6Address", "type": "string", "nullable": false, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": "ip"}, "interaction_micros": {"published_type": "int32", "type": "int32", "nullable": false, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}, "participated": {"published_type": "bool", "type": "bool", "nullable": false, "primary_key": false, "reference": null, "enum": null, "max_length": null, "format": null}}}}, "handling": {"timestamps": "UTC, microsecond precision; Spark timestamp in UTC session", "int64": "pandas nullable Int64; Spark LongType, never converted via float", "float64": "IEEE float64 as published, not silently converted to decimal", "uuid_ip": "Canonical validated string because Spark has no native UUID/IP scalar", "arrays": "Native array<string>, preserving allowed enum values", "strings": "No truncation. Published MaxLength validated; unbounded fields remain unbounded.", "withdrawal": "sim_support.course_withdrawal_events is supplemental, authoritative only inside this synthetic demonstration", "history": "sim_history.<table>_revisions adds effective_at, recorded_at, revision; not official Canvas columns", "gradebook": "sim_support.gradebook_qualification certifies finished course totals; scores.final_score alone is not a label", "logs": "Includes categorized anonymous/external requests. Real CD2 logs retain only 30 days; historical use needs archive.", "overrides": "ADHOC student overrides only; set_id null; mapping through assignment_override_students", "grades": "Point-weighted totals; no assignment-group weighting, dropped scores or grading periods", "current_term": "Official source and history stop at current cutoff, no term-five future truth is persisted"}}')
BUNDLE_INFO = json.loads('{"purpose": "academic_early_warning_synthetic_input", "filename": "academic-early-warning-simulated-data.zip", "profile": "small", "synthetic": true, "zip_sha256": "b162ff813446dbcffa17384d44cb59a25fa9574a2032b5609c625bb8040faf67", "zip_bytes": 15645570, "manifest_sha256": "b0e5bdfecf9306cacc53eb92778d7f15c986a4c4ceb3065e5bc4311d479efbfb", "member_count": 117, "uncompressed_bytes": 24619679, "seed": 271828, "counts": {"accounts": 1, "users": 408, "enrollment_terms": 5, "courses": 40, "course_sections": 120, "enrollments": 3040, "assignment_groups": 40, "assignments": 800, "assignment_overrides": 4611, "assignment_override_students": 4611, "submissions": 60000, "scores": 2400, "web_logs": 111217}, "historical_features": 2336, "current_features": 583, "expected_lakehouse_path": "Files/uploads/academic-early-warning-simulated-data.zip", "prepared_features_included": true, "pretrained_model_included": false, "parquet_timestamp_unit": "microseconds", "parquet_timestamp_metadata": "microseconds"}')
# Set this to the exact uploaded root, including '-fixed' if that is your folder name.
# It must directly contain source_parquet/, features/, manifest.json and summary.json.
SOURCE_FOLDER = 'Files/academic-early-warning-simulated-data'


In [ ]:
from aews.table_import import load_folder_tables
table_results = load_folder_tables(spark, SOURCE_CONFIG, SOURCE_CONTRACT, SOURCE_FOLDER, BUNDLE_INFO)


In [ ]:
import pandas as pd
display(pd.DataFrame(table_results))
